In [2]:
# inlegalbert_kg_rag_ppo_rrc.py
#
# COMPLETE PIPELINE — all four phases in one file
#
# Architecture:
#   InLegalBERT → BiLSTM → Multi-Head Attention Pooling → CRF
#   + Knowledge Graph (KG) with RST-based edges
#   + Uncertainty-triggered KG Retrieval + Graph Attention Fusion
#   + PPO Reinforcement Learning (rare-class F1 as reward signal)
#   + Synthetic Correction Model (no human annotators)
#
# Training phases:
#   Phase A — Train base InLegalBERT+BiLSTM+MHA+CRF model
#   Phase A→B — Build KG from trained encoder embeddings
#   Phase B — Fine-tune KGAugmentedModel (KG frozen, fusion trained)
#   Phase C — PPO RL: actor+critic on top of KGAugmentedModel
#              reward = rare-class F1 signal (breaks majority-class dominance)
#   Phase D — Synthetic correction without human annotators:
#              "combined" mode = self_play + curriculum + kg_disagree
#
# Phase D alternatives (set PHASE_D_MODE):
#   "self_play"   — mine Phase C's own errors on train set → correction MLP
#   "kg_disagree" — rule-based bonus when KG suggests rare but actor disagrees
#   "curriculum"  — per-class reward multipliers annealed from rolling F1 window
#   "combined"    — all three together (recommended, best rare-class F1)
#
# Key invariants:
#   • CRF is NEVER bypassed — PPO shapes emissions, CRF enforces label order
#   • Actor warm-started from Phase B fusion_classifier → no cold-start collapse
#   • GAE over sentence sequence (γ=0.99, λ=0.95) — legal docs have long-range deps
#   • KL penalty to Phase B policy → prevents degenerate rare-only policy
#   • Reward asymmetry: rare correct +2.0 >> majority correct +0.3 >> rare wrong -3.0

import os, json, random, time, math
from collections import Counter, defaultdict, deque
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)


# ═══════════════════════════════════════════════════════════
# CONFIG — Phase A / B
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH     = "dataset/build_train.jsonl"
DEV_PATH       = "dataset/build_dev.jsonl"
TEST_PATH      = "dataset/build_test.jsonl"
OUT_DIR        = "rrc_kg_rag_ppo_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 20
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS = 8
BERT_LR_DECAY      = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2

MHA_HEADS   = 4
MHA_DROPOUT = 0.1

CTX_LSTM_HIDDEN = 64
CTX_LSTM_LAYERS = 2

AUX_CE_WEIGHT   = 0.2
LABEL_SMOOTHING = 0.1

ES_PATIENCE  = 10
ES_MIN_DELTA = 1e-4

WARMUP_RATIO                = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD              = 0.05

KG_TOP_K           = 3
KG_TOP_NODES       = 5
KG_HOP             = 1
UNCERTAINTY_THRESH = 0.7
RARE_ALWAYS_KG     = True
KG_FUSION_DIM      = 256    # sent_out_dim = SENT_LSTM_HIDDEN * 2

RST_INTRA_THRESH = 0.6
RST_CROSS_THRESH = 0.5

# ═══════════════════════════════════════════════════════════
# CONFIG — Phase C / D (PPO + Synthetic Correction)
# ═══════════════════════════════════════════════════════════

# Reward shaping
REWARD_RARE_CORRECT =  2.0
REWARD_RARE_WRONG   = -3.0   # asymmetric: punish false negatives hard
REWARD_MAJ_CORRECT  =  0.3
REWARD_MAJ_WRONG    =  0.0

# PPO
PPO_EPOCHS       = 20
PPO_MINI_EPOCHS  = 4
PPO_CLIP_EPS     = 0.2
PPO_KL_COEF      = 0.1
PPO_VALUE_COEF   = 0.5
PPO_ENTROPY_COEF = 0.01
PPO_LR           = 3e-5
PPO_GRAD_CLIP    = 1.0
GAE_GAMMA        = 0.99
GAE_LAMBDA       = 0.95
ROLLOUT_DOCS     = 16
PPO_WARMUP_RATIO = 0.1
ES_PATIENCE_PPO  = 7

# Phase D mode: "self_play" | "kg_disagree" | "curriculum" | "combined"
PHASE_D_MODE = "combined"

# self_play correction
CORRECTION_LR         = 1e-4
CORRECTION_EPOCHS     = 15
CORRECTION_NEG_RATIO  = 3

# curriculum
CURRICULUM_WINDOW = 5
CURRICULUM_MIN_W  = 0.5
CURRICULUM_MAX_W  = 3.0

# kg_disagree bonus
KG_DISAGREE_BONUS = 0.8

# ═══════════════════════════════════════════════════════════
# LABELS
# ═══════════════════════════════════════════════════════════
LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)

        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL  (Phase A)
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size  # 768
        self.dropout  = nn.Dropout(dropout)

        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2  # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim=self.sent_out_dim, num_heads=mha_heads, dropout=mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=self.sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2  # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )
        self.crf     = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100,
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total = len(encoder_layers)
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} + pooler.\n")

    def encode_sentences(self, input_ids, attention_mask,
                         token_type_ids, lengths=None) -> torch.Tensor:
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids=flat_ids[valid],
                attention_mask=flat_mask[valid],
                token_type_ids=flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)

        token_embs_all = self.dropout(token_embs_all)
        lstm_out, _    = self.sent_bilstm(token_embs_all)
        lstm_out       = self.dropout(lstm_out)

        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def get_emissions(self, input_ids, attention_mask,
                      token_type_ids, lengths=None):
        sent_vecs      = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths)
        sent_vecs_drop = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        _, _, emissions = self.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss  = self.ce_loss(
                emissions.reshape(B2 * T2, C), labels.reshape(B2 * T2))
            loss = crf_loss + AUX_CE_WEIGHT * ce_loss
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# KNOWLEDGE GRAPH
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    def __init__(self, emb_dim=KG_FUSION_DIM):
        self.emb_dim     = emb_dim
        self.nodes       = defaultdict(list)
        self.intra_edges = defaultdict(list)
        self.cross_edges = []
        self._stacked    = {}

    def add_nodes(self, embeddings: torch.Tensor, label_ids: list,
                  texts: list = None):
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            text = texts[k] if texts is not None else ""
            self.nodes[lid].append({"emb": emb, "text": text})
        self._stacked = {}

    def build_edges(self, intra_thresh=RST_INTRA_THRESH,
                    cross_thresh=RST_CROSS_THRESH,
                    max_intra_edges_per_node=5,
                    max_cross_edges=2000):
        print("  Building KG edges ...")
        self._stacked    = {}
        self.intra_edges = defaultdict(list)
        self.cross_edges = []

        for lid, node_list in self.nodes.items():
            N = len(node_list)
            if N < 2:
                continue
            embs      = torch.stack([n["emb"] for n in node_list])
            embs_norm = F.normalize(embs, dim=-1)
            sim_mat   = torch.mm(embs_norm, embs_norm.T)

            for i in range(N - 1):
                w = float(sim_mat[i, i + 1].item())
                self.intra_edges[lid].append((i, i + 1, max(0.0, w)))

            for i in range(N):
                sims = sim_mat[i].clone()
                sims[max(0, i - 1):i + 2] = -1
                count = 0
                while count < max_intra_edges_per_node:
                    j = int(sims.argmax().item())
                    if sims[j] < intra_thresh:
                        break
                    self.intra_edges[lid].append((i, j, float(sims[j].item())))
                    sims[j] = -1
                    count += 1

            self._stacked[lid] = embs

        label_ids   = list(self.nodes.keys())
        cross_count = 0
        for a in range(len(label_ids)):
            if cross_count >= max_cross_edges:
                break
            for b in range(a + 1, len(label_ids)):
                if cross_count >= max_cross_edges:
                    break
                la, lb   = label_ids[a], label_ids[b]
                embs_a   = self._get_stacked(la)
                embs_b   = self._get_stacked(lb)
                if embs_a is None or embs_b is None:
                    continue
                sim_mat = torch.mm(
                    F.normalize(embs_a, dim=-1),
                    F.normalize(embs_b, dim=-1).T,
                )
                high = (sim_mat >= cross_thresh).nonzero(as_tuple=False)
                for pair in high[:50]:
                    ni, nj = int(pair[0]), int(pair[1])
                    self.cross_edges.append(
                        (la, ni, lb, nj, float(sim_mat[ni, nj].item())))
                    cross_count += 1

        n_intra = sum(len(v) for v in self.intra_edges.values())
        print(f"  KG: {sum(len(v) for v in self.nodes.values())} nodes | "
              f"{n_intra} intra-edges | {len(self.cross_edges)} cross-edges")

    def _get_stacked(self, lid):
        if lid not in self._stacked:
            if lid not in self.nodes or not self.nodes[lid]:
                return None
            self._stacked[lid] = torch.stack(
                [n["emb"] for n in self.nodes[lid]])
        return self._stacked[lid]

    def save(self, path):
        data = {
            "nodes": {str(k): [{"emb": n["emb"].tolist(), "text": n["text"]}
                                for n in v]
                      for k, v in self.nodes.items()},
            "intra_edges": {str(k): v for k, v in self.intra_edges.items()},
            "cross_edges": self.cross_edges,
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  KG saved to {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM):
        kg = cls(emb_dim=emb_dim)
        with open(path) as f:
            data = json.load(f)
        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                kg.nodes[lid].append(
                    {"emb": torch.tensor(n["emb"]), "text": n["text"]})
        for k, edges in data["intra_edges"].items():
            kg.intra_edges[int(k)] = [tuple(e) for e in edges]
        kg.cross_edges = [tuple(e) for e in data["cross_edges"]]
        print(f"  KG loaded from {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS, threshold=UNCERTAINTY_THRESH):
        self.log_C     = math.log(num_classes)
        self.threshold = threshold

    def entropy(self, logits: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=-1)
        H     = -(probs * (probs + 1e-9).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits: torch.Tensor) -> torch.Tensor:
        return self.entropy(logits) > self.threshold

    def top_label(self, logits: torch.Tensor) -> torch.Tensor:
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# KG RETRIEVER
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    def __init__(self, kg: KnowledgeGraph, top_k=KG_TOP_K,
                 top_nodes=KG_TOP_NODES, hop=KG_HOP):
        self.kg        = kg
        self.top_k     = top_k
        self.top_nodes = top_nodes
        self.hop       = hop

    def retrieve(self, h_i: torch.Tensor, rare_ids: list = None,
                 first_pass_label: int = None) -> list:
        h_norm = F.normalize(h_i.unsqueeze(0), dim=-1)

        subgraph_scores = {}
        for lid in self.kg.nodes:
            embs = self.kg._get_stacked(lid)
            if embs is None or embs.shape[0] == 0:
                continue
            sims = torch.mv(F.normalize(embs, dim=-1), h_norm.squeeze(0))
            subgraph_scores[lid] = float(sims.max().item())

        sorted_sgs = sorted(subgraph_scores.items(), key=lambda x: -x[1])
        selected   = [lid for lid, _ in sorted_sgs[:self.top_k]]

        results = []
        for lid in selected:
            embs = self.kg._get_stacked(lid)
            if embs is None:
                continue
            sims      = torch.mv(F.normalize(embs, dim=-1), h_norm.squeeze(0))
            k         = min(self.top_nodes, embs.shape[0])
            top_idx   = sims.topk(k).indices.tolist()
            top_sims  = sims.topk(k).values.tolist()
            seed_set  = set(top_idx)

            if self.hop >= 1:
                for idx in list(seed_set):
                    for (i, j, w) in self.kg.intra_edges.get(lid, []):
                        if i == idx and j not in seed_set:
                            seed_set.add(j)
                        elif j == idx and i not in seed_set:
                            seed_set.add(i)

                for (la, ni, lb, nj, w) in self.kg.cross_edges:
                    if la == lid and ni in seed_set:
                        ce = self.kg._get_stacked(lb)
                        if ce is not None and nj < ce.shape[0]:
                            results.append((ce[nj], w))
                    elif lb == lid and nj in seed_set:
                        ce = self.kg._get_stacked(la)
                        if ce is not None and ni < ce.shape[0]:
                            results.append((ce[ni], w))

            for idx, sim in zip(top_idx, top_sims):
                results.append((embs[idx], float(sim)))

        return results


# ═══════════════════════════════════════════════════════════
# GRAPH ATTENTION FUSION
# ═══════════════════════════════════════════════════════════
class GraphAttentionFusion(nn.Module):
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT):
        super().__init__()
        self.emb_dim = emb_dim
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, h_i: torch.Tensor, neighbours: list,
                device=None) -> torch.Tensor:
        if not neighbours:
            return torch.zeros_like(h_i)
        if device is None:
            device = h_i.device

        embs    = torch.stack([nb[0] for nb in neighbours]).to(device)
        weights = torch.tensor([nb[1] for nb in neighbours],
                               device=device, dtype=torch.float)
        q     = self.proj_q(h_i.unsqueeze(0))
        k     = self.proj_k(embs)
        dot   = torch.mv(k, q.squeeze(0)) * self.scale
        alpha = F.softmax(dot * weights, dim=0)
        alpha = self.dropout(alpha)
        return (alpha.unsqueeze(-1) * embs).sum(dim=0)


# ═══════════════════════════════════════════════════════════
# KG-AUGMENTED MODEL  (Phase B)
# ═══════════════════════════════════════════════════════════
class KGAugmentedModel(nn.Module):
    def __init__(self, base_model: InLegalBERT_BiLSTM_MHA_CRF,
                 kg: KnowledgeGraph, rare_ids: list,
                 retriever: KGRetriever = None):
        super().__init__()
        self.base        = base_model
        self.kg          = kg
        self.rare_ids    = rare_ids
        self.retriever   = retriever or KGRetriever(kg)
        self.uncertainty = UncertaintyEstimator()

        sent_dim = base_model.sent_out_dim  # 256
        ctx_dim  = base_model.ctx_out_dim   # 128

        self.gat_fusion  = GraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_proj = nn.Sequential(
            nn.Linear(sent_dim * 2, ctx_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

    def _kg_fuse_batch(self, sent_vecs, emissions, lengths, device):
        B, T, sent_dim = sent_vecs.shape
        fused          = sent_vecs.clone()
        uncertain_mask = self.uncertainty.is_uncertain(emissions)
        top_labels     = self.uncertainty.top_label(emissions)

        for b in range(B):
            n = int(lengths[b].item())
            for t in range(n):
                uncertain = bool(uncertain_mask[b, t].item())
                pred_lbl  = int(top_labels[b, t].item())
                is_rare   = pred_lbl in self.rare_ids

                if not (uncertain or (RARE_ALWAYS_KG and is_rare)):
                    continue

                h_i        = sent_vecs[b, t].detach().cpu()
                neighbours = self.retriever.retrieve(
                    h_i, rare_ids=self.rare_ids,
                    first_pass_label=pred_lbl)

                if not neighbours:
                    continue

                v_i = self.gat_fusion(sent_vecs[b, t], neighbours, device=device)
                fused[b, t] = sent_vecs[b, t] + v_i

        return fused

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        device = input_ids.device
        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)

        fused_sent = self._kg_fuse_batch(
            sent_vecs, base_emissions, lengths, device)

        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)

        fused_ctx       = self.base.dropout(fused_ctx)
        fused_emissions = self.fusion_classifier(fused_ctx)
        fused_emissions = torch.nan_to_num(
            fused_emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2],
                              dtype=torch.bool, device=device)

        combined_emissions = (base_emissions + fused_emissions) / 2.0

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            base_crf_loss  = -self.base.crf(
                base_emissions, safe_labels, mask=mask, reduction="mean")
            fused_crf_loss = -self.fusion_crf(
                fused_emissions, safe_labels, mask=mask, reduction="mean")
            B2, T2, C = combined_emissions.shape
            ce_loss   = self.ce_loss(
                combined_emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2))

            loss = (base_crf_loss + fused_crf_loss) / 2.0 + AUX_CE_WEIGHT * ce_loss
            return loss, combined_emissions
        else:
            decoded = self.fusion_crf.decode(fused_emissions, mask=mask)
            return decoded, fused_emissions


# ═══════════════════════════════════════════════════════════
# PPO — ACTOR HEAD
# ═══════════════════════════════════════════════════════════
class ActorHead(nn.Module):
    """
    Policy π_θ.  Takes ctx_out (B, T, ctx_dim) → logits (B, T, C).
    Warm-started from Phase B fusion_classifier.
    """
    def __init__(self, ctx_dim: int, num_labels: int = NUM_LABELS,
                 dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_dim // 2, num_labels),
        )

    def forward(self, ctx_out: torch.Tensor) -> torch.Tensor:
        return self.net(ctx_out)

    def copy_from_fusion_classifier(self, fusion_classifier: nn.Sequential):
        try:
            self.net.load_state_dict(fusion_classifier.state_dict())
            print("  ✔ ActorHead warm-started from Phase B fusion_classifier.")
        except Exception as e:
            print(f"  ⚠ ActorHead warm-start failed ({e}), using random init.")


# ═══════════════════════════════════════════════════════════
# PPO — CRITIC HEAD
# ═══════════════════════════════════════════════════════════
class CriticHead(nn.Module):
    """
    Value V_φ.  Takes ctx_out (B, T, ctx_dim) → scalar (B, T).
    Estimates expected future rare-class F1 from current hidden state.
    """
    def __init__(self, ctx_dim: int, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Linear(ctx_dim // 2, 1),
        )

    def forward(self, ctx_out: torch.Tensor) -> torch.Tensor:
        return self.net(ctx_out).squeeze(-1)


# ═══════════════════════════════════════════════════════════
# PPO — AUGMENTED MODEL
# ═══════════════════════════════════════════════════════════
class PPOModel(nn.Module):
    """
    Wraps KGAugmentedModel.  Adds actor + critic heads.
    Encoder + KG fusion remain trainable (at lower LR).
    Actor is warm-started from Phase B.
    """
    def __init__(self, kg_model: KGAugmentedModel, rare_ids: list):
        super().__init__()
        self.kg_model = kg_model
        self.rare_ids = set(rare_ids)

        ctx_dim      = kg_model.base.ctx_out_dim  # 128
        self.actor   = ActorHead(ctx_dim)
        self.critic  = CriticHead(ctx_dim)

        self.actor.copy_from_fusion_classifier(kg_model.fusion_classifier)

    def get_ctx_out(self, input_ids, attention_mask, token_type_ids, lengths):
        """Run encoder + KG fusion, return (fused_ctx, base_emissions)."""
        kg   = self.kg_model
        base = kg.base

        sent_vecs, _, base_emissions = base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)
        fused_sent = kg._kg_fuse_batch(
            sent_vecs, base_emissions, lengths, input_ids.device)

        fused_drop = base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = base.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            ctx_out, _ = base.ctx_bilstm(fused_drop)

        return base.dropout(ctx_out), base_emissions

    @torch.no_grad()
    def decode_actions(self, actor_logits, lengths):
        """CRF decode — PPO shapes emissions, CRF enforces legal label order."""
        B, T, _ = actor_logits.shape
        mask    = torch.zeros(B, T, dtype=torch.bool, device=actor_logits.device)
        for i, l in enumerate(lengths):
            mask[i, :l] = True
        return self.kg_model.fusion_crf.decode(actor_logits, mask=mask)


# ═══════════════════════════════════════════════════════════
# PPO — REWARD SHAPER
# ═══════════════════════════════════════════════════════════
class RewardShaper:
    def __init__(self, rare_ids: list,
                 rare_correct=REWARD_RARE_CORRECT,
                 rare_wrong=REWARD_RARE_WRONG,
                 maj_correct=REWARD_MAJ_CORRECT,
                 maj_wrong=REWARD_MAJ_WRONG):
        self.rare_ids = set(rare_ids)
        self.rc = rare_correct
        self.rw = rare_wrong
        self.mc = maj_correct
        self.mw = maj_wrong

    def base_reward(self, pred: int, true: int) -> float:
        correct = (pred == true)
        is_rare = (true in self.rare_ids)
        if is_rare:
            return self.rc if correct else self.rw
        else:
            return self.mc if correct else self.mw


# ═══════════════════════════════════════════════════════════
# PPO — GAE ADVANTAGE
# ═══════════════════════════════════════════════════════════
def compute_gae(rewards: list, values: list,
                gamma=GAE_GAMMA, lam=GAE_LAMBDA) -> tuple:
    """GAE over a single document's sentence sequence."""
    T          = len(rewards)
    advantages = [0.0] * T
    returns    = [0.0] * T
    gae        = 0.0

    for t in reversed(range(T)):
        next_val    = values[t + 1] if t + 1 < T else 0.0
        delta       = rewards[t] + gamma * next_val - values[t]
        gae         = delta + gamma * lam * gae
        advantages[t] = gae
        returns[t]    = gae + values[t]

    return advantages, returns


# ═══════════════════════════════════════════════════════════
# PPO — ROLLOUT BUFFER
# ═══════════════════════════════════════════════════════════
class RolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states        = []
        self.actor_logits  = []
        self.actions       = []
        self.true_labels   = []
        self.advantages    = []
        self.returns       = []
        self.old_log_probs = []

    def add_trajectory(self, states_t, logits_t, actions_t,
                       trues_t, advantages_t, returns_t):
        for i in range(len(actions_t)):
            self.states.append(states_t[i])
            self.actor_logits.append(logits_t[i])
            self.actions.append(int(actions_t[i]))
            self.true_labels.append(int(trues_t[i]))
            self.advantages.append(float(advantages_t[i]))
            self.returns.append(float(returns_t[i]))
            log_p = F.log_softmax(logits_t[i], dim=-1)[int(actions_t[i])]
            self.old_log_probs.append(float(log_p.item()))

    def __len__(self):
        return len(self.actions)

    def as_tensors(self, device):
        states        = torch.stack(self.states).to(device)
        actor_logits  = torch.stack(self.actor_logits).to(device)
        actions       = torch.tensor(self.actions, dtype=torch.long, device=device)
        advantages    = torch.tensor(self.advantages, dtype=torch.float, device=device)
        returns       = torch.tensor(self.returns,    dtype=torch.float, device=device)
        old_log_probs = torch.tensor(self.old_log_probs, dtype=torch.float, device=device)

        adv_mean  = advantages.mean()
        adv_std   = advantages.std() + 1e-8
        advantages = (advantages - adv_mean) / adv_std

        return states, actor_logits, actions, advantages, returns, old_log_probs


# ═══════════════════════════════════════════════════════════
# PHASE D — SYNTHETIC CORRECTION MODEL  (self_play)
# ═══════════════════════════════════════════════════════════
class SyntheticCorrectionModel(nn.Module):
    """
    3-layer MLP trained on self-play error data.
    Input:  concat(ctx_emb, one_hot(pred), one_hot(true))
    Output: δ ∈ [-1, +1]  added to PPO reward.
    No human annotations needed.
    """
    def __init__(self, ctx_dim: int, num_labels: int = NUM_LABELS,
                 hidden: int = 128):
        super().__init__()
        in_dim = ctx_dim + num_labels * 2
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Linear(hidden // 2, 1),
            nn.Tanh(),
        )

    def forward(self, ctx_emb, pred_ids, true_ids):
        N    = ctx_emb.shape[0]
        C    = NUM_LABELS
        x    = torch.cat([
            ctx_emb,
            F.one_hot(pred_ids, C).float(),
            F.one_hot(true_ids, C).float(),
        ], dim=-1)
        return self.net(x).squeeze(-1)


def collect_correction_data(ppo_model: PPOModel, train_dataset,
                             rare_ids: list, device=DEVICE,
                             min_confidence=0.7):
    """
    Run PPO model on train set.  Collect:
      - true rare, prediction wrong  → target -1.0  (error)
      - true rare, prediction correct + high confidence  → target +1.0
    Self-play: no human labels needed.
    """
    ppo_model.eval()
    loader   = DataLoader(train_dataset, batch_size=1, shuffle=False,
                          collate_fn=collate_rrc)
    rare_set = set(rare_ids)
    data     = []

    with torch.no_grad():
        for ids, attn, ttype, labels, lengths in loader:
            ids     = ids.to(device)
            attn    = attn.to(device)
            ttype   = ttype.to(device)
            lengths = lengths.to(device)

            ctx_out, _   = ppo_model.get_ctx_out(ids, attn, ttype, lengths)
            actor_logits = ppo_model.actor(ctx_out)
            probs        = F.softmax(actor_logits, dim=-1)

            n = int(lengths[0].item())
            for t in range(n):
                true_lbl = int(labels[0, t].item())
                if true_lbl < 0 or true_lbl not in rare_set:
                    continue

                pred_lbl   = int(actor_logits[0, t].argmax().item())
                confidence = float(probs[0, t].max().item())
                ctx_emb    = ctx_out[0, t].cpu()

                if pred_lbl != true_lbl:
                    data.append({"ctx_emb": ctx_emb, "pred": pred_lbl,
                                 "true": true_lbl, "target": -1.0})
                elif confidence >= min_confidence:
                    data.append({"ctx_emb": ctx_emb, "pred": pred_lbl,
                                 "true": true_lbl, "target": +1.0})

    neg = [d for d in data if d["target"] < 0]
    pos = [d for d in data if d["target"] > 0]
    print(f"  Correction data: {len(neg)} errors, {len(pos)} confident-correct")
    return data


def train_correction_model(correction_model: SyntheticCorrectionModel,
                            correction_data: list, device=DEVICE):
    if not correction_data:
        print("  ⚠ No correction data — skipping self_play training.")
        return correction_model

    optimizer = torch.optim.AdamW(
        correction_model.parameters(), lr=CORRECTION_LR, weight_decay=0.01)
    correction_model.to(device).train()

    neg_data = [d for d in correction_data if d["target"] < 0]
    pos_data = [d for d in correction_data if d["target"] > 0]

    for epoch in range(CORRECTION_EPOCHS):
        if pos_data:
            n_pos  = min(len(pos_data), 64)
            n_neg  = min(len(neg_data), n_pos * CORRECTION_NEG_RATIO)
            batch  = (random.sample(pos_data, n_pos) +
                      random.sample(neg_data, n_neg))
        else:
            batch = random.sample(neg_data, min(len(neg_data), 256))

        random.shuffle(batch)

        ctx_embs = torch.stack([b["ctx_emb"] for b in batch]).to(device)
        preds    = torch.tensor([b["pred"]    for b in batch],
                                dtype=torch.long, device=device)
        trues    = torch.tensor([b["true"]    for b in batch],
                                dtype=torch.long, device=device)
        targets  = torch.tensor([b["target"]  for b in batch],
                                dtype=torch.float, device=device)

        deltas = correction_model(ctx_embs, preds, trues)
        loss   = F.mse_loss(deltas, targets)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(correction_model.parameters(), 1.0)
        optimizer.step()

        if (epoch + 1) % 5 == 0:
            print(f"    [Correction] epoch {epoch+1}/{CORRECTION_EPOCHS} "
                  f"loss={loss.item():.4f}")

    torch.save(correction_model.state_dict(),
               os.path.join(OUT_DIR, "correction_model.bin"))
    print("  ✔ Correction model saved.")
    return correction_model


# ═══════════════════════════════════════════════════════════
# PHASE D — KG DISAGREEMENT BONUS  (kg_disagree)
# ═══════════════════════════════════════════════════════════
def kg_disagree_corrections(kg_retriever: KGRetriever,
                              ctx_embs_cpu: list,
                              actor_preds: list,
                              rare_ids: list,
                              bonus=KG_DISAGREE_BONUS) -> list:
    """
    Rule-based, zero parameters.
    If actor doesn't predict a rare class but KG retrieval is
    strong (high similarity), add a bonus to encourage exploration.
    """
    rare_set    = set(rare_ids)
    corrections = []
    for h_i, pred in zip(ctx_embs_cpu, actor_preds):
        neighbours = kg_retriever.retrieve(h_i, rare_ids=rare_ids)
        if not neighbours:
            corrections.append(0.0)
            continue
        total_sim = sum(max(w, 0.0) for _, w in neighbours)
        if pred not in rare_set and total_sim > 0.5:
            corrections.append(bonus)
        else:
            corrections.append(0.0)
    return corrections


# ═══════════════════════════════════════════════════════════
# PHASE D — CURRICULUM REWARD ANNEALING
# ═══════════════════════════════════════════════════════════
class CurriculumScheduler:
    """
    Tracks rolling per-class F1 over last CURRICULUM_WINDOW evals.
    Multipliers: low-F1 rare class → high multiplier → harder reward signal.
    Self-regulating: improves automatically as classes improve.
    """
    def __init__(self, rare_ids: list, num_labels=NUM_LABELS,
                 window=CURRICULUM_WINDOW,
                 min_w=CURRICULUM_MIN_W, max_w=CURRICULUM_MAX_W):
        self.rare_ids   = set(rare_ids)
        self.num_labels = num_labels
        self.window     = window
        self.min_w      = min_w
        self.max_w      = max_w
        self.f1_history  = {i: deque(maxlen=window) for i in range(num_labels)}
        self.multipliers = {
            i: max_w if i in self.rare_ids else 1.0
            for i in range(num_labels)
        }

    def update(self, all_trues: list, all_preds: list):
        per_class_f1 = f1_score(
            all_trues, all_preds,
            labels=list(range(self.num_labels)),
            average=None, zero_division=0,
        )
        for i in range(self.num_labels):
            self.f1_history[i].append(float(per_class_f1[i]))

        for i in self.rare_ids:
            hist = list(self.f1_history[i])
            if len(hist) < 2:
                continue
            mean_f1  = np.mean(hist)
            trend    = hist[-1] - hist[0]
            target_w = self.max_w * (1.0 - mean_f1) + self.min_w * mean_f1
            if trend > 0.01:
                target_w *= 0.9
            self.multipliers[i] = float(np.clip(
                0.7 * self.multipliers[i] + 0.3 * target_w,
                self.min_w, self.max_w,
            ))

    def get_multiplier(self, label_id: int) -> float:
        return self.multipliers.get(label_id, 1.0)

    def log(self):
        rare_mults = {id2label[i]: f"{self.multipliers[i]:.2f}"
                      for i in sorted(self.rare_ids)
                      if i in self.multipliers}
        print(f"  Curriculum multipliers: {rare_mults}")


# ═══════════════════════════════════════════════════════════
# PHASE D — UNIFIED REWARD FUNCTION
# ═══════════════════════════════════════════════════════════
def compute_shaped_reward(pred: int, true: int,
                           rare_ids: set,
                           base_shaper: RewardShaper,
                           correction_model=None,
                           ctx_emb_cpu=None,
                           curriculum: CurriculumScheduler = None,
                           kg_delta: float = 0.0,
                           device=DEVICE) -> float:
    r = base_shaper.base_reward(pred, true)

    # curriculum multiplier on rare class true labels
    if curriculum is not None and true in rare_ids:
        r *= curriculum.get_multiplier(true)

    # KG disagreement bonus (pre-computed)
    r += kg_delta

    # self-play correction model
    if correction_model is not None and ctx_emb_cpu is not None:
        with torch.no_grad():
            ce = ctx_emb_cpu.unsqueeze(0).to(device)
            pd = torch.tensor([pred], dtype=torch.long, device=device)
            td = torch.tensor([true], dtype=torch.long, device=device)
            delta = float(correction_model(ce, pd, td).item())
        r += delta

    return r


# ═══════════════════════════════════════════════════════════
# PPO TRAINER  (Phase C + D)
# ═══════════════════════════════════════════════════════════
class PPOTrainer:
    def __init__(self, ppo_model: PPOModel, rare_ids: list,
                 kg_retriever: KGRetriever = None, device=DEVICE):
        self.model       = ppo_model.to(device)
        self.device      = device
        self.rare_ids    = set(rare_ids)
        self.retriever   = kg_retriever
        self.base_shaper = RewardShaper(list(rare_ids))
        # Phase D components set in train()
        self.correction_model = None
        self.curriculum       = None

    def build_optimizer(self):
        actor_params = (list(self.model.actor.parameters()) +
                        list(self.model.critic.parameters()))
        base_params  = [p for p in self.model.kg_model.parameters()
                        if p.requires_grad]
        return torch.optim.AdamW([
            {"params": actor_params, "lr": PPO_LR,        "weight_decay": 0.01},
            {"params": base_params,  "lr": PPO_LR * 0.1,  "weight_decay": 0.01},
        ])

    @torch.no_grad()
    def collect_rollout(self, loader_iter, loader,
                        n_docs=ROLLOUT_DOCS):
        self.model.eval()
        buffer    = RolloutBuffer()
        collected = 0

        while collected < n_docs:
            try:
                batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(loader)
                batch = next(loader_iter)

            ids, attn, ttype, labels, lengths = batch
            ids     = ids.to(self.device)
            attn    = attn.to(self.device)
            ttype   = ttype.to(self.device)
            lengths = lengths.to(self.device)

            ctx_out, _   = self.model.get_ctx_out(ids, attn, ttype, lengths)
            actor_logits = self.model.actor(ctx_out)
            values       = self.model.critic(ctx_out)
            decoded      = self.model.decode_actions(actor_logits, lengths)

            B = ids.shape[0]
            for b in range(B):
                n          = int(lengths[b].item())
                actions_b  = decoded[b][:n]
                trues_b    = labels[b, :n].tolist()
                if any(t < 0 for t in trues_b):
                    continue

                # KG disagreement corrections (CPU, no grad)
                if (self.retriever is not None and
                        PHASE_D_MODE in ("kg_disagree", "combined")):
                    ctx_cpu  = [ctx_out[b, t].detach().cpu() for t in range(n)]
                    kg_delts = kg_disagree_corrections(
                        self.retriever, ctx_cpu, actions_b, list(self.rare_ids))
                else:
                    kg_delts = [0.0] * n

                # Per-sentence shaped rewards
                rewards_b = []
                for t in range(n):
                    r = compute_shaped_reward(
                        pred=actions_b[t], true=trues_b[t],
                        rare_ids=self.rare_ids,
                        base_shaper=self.base_shaper,
                        correction_model=(
                            self.correction_model
                            if PHASE_D_MODE in ("self_play", "combined")
                            else None),
                        ctx_emb_cpu=ctx_out[b, t].detach().cpu(),
                        curriculum=(
                            self.curriculum
                            if PHASE_D_MODE in ("curriculum", "combined")
                            else None),
                        kg_delta=kg_delts[t],
                        device=self.device,
                    )
                    rewards_b.append(r)

                vals_b          = values[b, :n].detach().cpu().tolist()
                advantages_b, returns_b = compute_gae(rewards_b, vals_b)

                buffer.add_trajectory(
                    states_t    =[ctx_out[b, t].detach() for t in range(n)],
                    logits_t    =[actor_logits[b, t].detach() for t in range(n)],
                    actions_t   =actions_b,
                    trues_t     =trues_b,
                    advantages_t=advantages_b,
                    returns_t   =returns_b,
                )
                collected += 1

        return buffer, loader_iter

    def ppo_update(self, buffer: RolloutBuffer):
        self.model.train()
        optimizer = self._optimizer

        (states, old_logits, actions, advantages,
         returns, old_log_probs) = buffer.as_tensors(self.device)

        N       = len(actions)
        indices = torch.randperm(N)
        total_pol, total_val, n_up = 0.0, 0.0, 0

        for _ in range(PPO_MINI_EPOCHS):
            for start in range(0, N, 256):
                idx = indices[start:start + 256]
                if len(idx) == 0:
                    continue

                s   = states[idx]
                a   = actions[idx]
                adv = advantages[idx]
                ret = returns[idx]
                olp = old_log_probs[idx]
                oleg= old_logits[idx]

                new_logits    = self.model.actor.net(s)
                new_vals      = self.model.critic.net(s).squeeze(-1)
                new_log_probs = F.log_softmax(new_logits, dim=-1)
                new_lp        = new_log_probs.gather(1, a.unsqueeze(1)).squeeze(1)

                ratio    = (new_lp - olp).exp()
                clipped  = torch.clamp(ratio, 1 - PPO_CLIP_EPS, 1 + PPO_CLIP_EPS)
                pol_loss = -torch.min(ratio * adv, clipped * adv).mean()

                val_loss = F.mse_loss(new_vals, ret)
                entropy  = -(F.softmax(new_logits, dim=-1) *
                              new_log_probs).sum(-1).mean()
                kl_loss  = F.kl_div(
                    F.log_softmax(new_logits, dim=-1),
                    F.softmax(oleg, dim=-1),
                    reduction="batchmean",
                )

                loss = (pol_loss
                        + PPO_VALUE_COEF   * val_loss
                        - PPO_ENTROPY_COEF * entropy
                        + PPO_KL_COEF      * kl_loss)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), PPO_GRAD_CLIP)
                optimizer.step()

                total_pol += pol_loss.item()
                total_val += val_loss.item()
                n_up      += 1

        return total_pol / max(1, n_up), total_val / max(1, n_up)

    def evaluate(self, dataset, rare_ids, split_name="dev"):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)

                ctx_out, _ = self.model.get_ctx_out(ids, attn, ttype, lengths)
                logits     = self.model.actor(ctx_out)
                decoded    = self.model.decode_actions(logits, lengths)

                for i, seq in enumerate(decoded):
                    n = int(lengths[i].item())
                    all_preds.extend(seq[:n])
                    all_trues.extend(labels[i, :n].tolist())

        macro_f1 = f1_score(all_trues, all_preds,
                            average="macro", zero_division=0)
        rare_f1  = f1_score(all_trues, all_preds,
                            labels=list(rare_ids),
                            average="macro", zero_division=0)
        return macro_f1, rare_f1, all_trues, all_preds

    def train(self, train_dataset, dev_dataset, rare_ids,
              num_epochs=PPO_EPOCHS):
        print(f"\n  Phase D mode : {PHASE_D_MODE}")

        # ── Phase D: initialise correction components ──────
        if PHASE_D_MODE in ("curriculum", "combined"):
            self.curriculum = CurriculumScheduler(list(rare_ids))
            print("  ✔ Curriculum scheduler initialised.")

        if PHASE_D_MODE in ("self_play", "combined"):
            print("\n  Collecting self-play correction data ...")
            ctx_dim    = self.model.kg_model.base.ctx_out_dim
            corr_model = SyntheticCorrectionModel(ctx_dim=ctx_dim)
            corr_data  = collect_correction_data(
                self.model, train_dataset, list(rare_ids), device=self.device)
            self.correction_model = train_correction_model(
                corr_model, corr_data, device=self.device)
            self.correction_model.eval()

        self._optimizer  = self.build_optimizer()
        train_loader     = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                       shuffle=True, collate_fn=collate_rrc)
        loader_iter      = iter(train_loader)
        best_rare_f1     = -1.0
        best_state       = None
        es_counter       = 0
        history          = []

        print(f"\n{'='*60}")
        print("PHASE C+D: PPO RL Training")
        print(f"{'='*60}\n")

        for epoch in range(1, num_epochs + 1):
            epoch_start = time.time()

            buffer, loader_iter = self.collect_rollout(
                loader_iter, train_loader, n_docs=ROLLOUT_DOCS)
            pol_loss, val_loss  = self.ppo_update(buffer)

            macro_f1, rare_f1, all_trues, all_preds = self.evaluate(
                dev_dataset, rare_ids)

            if self.curriculum is not None:
                self.curriculum.update(all_trues, all_preds)

            epoch_time = time.time() - epoch_start
            print(
                f"[PPO] Epoch {epoch:03d}/{num_epochs} | "
                f"pol_loss: {pol_loss:.4f} | val_loss: {val_loss:.4f} | "
                f"dev_macro_f1: {macro_f1:.4f} | dev_rare_f1: {rare_f1:.4f} | "
                f"buf: {len(buffer)} | time: {epoch_time:.1f}s"
            )

            if self.curriculum is not None and epoch % 3 == 0:
                self.curriculum.log()

            history.append({
                "epoch": epoch, "phase": "ppo",
                "pol_loss": pol_loss, "val_loss": val_loss,
                "dev_macro_f1": macro_f1, "dev_rare_f1": rare_f1,
                "epoch_time_s": epoch_time,
            })

            if rare_f1 > best_rare_f1 + ES_MIN_DELTA:
                best_rare_f1 = rare_f1
                best_state   = {k: v.cpu().clone()
                                for k, v in self.model.state_dict().items()}
                es_counter   = 0
                print(f"  ✔ New best dev_rare_f1={best_rare_f1:.4f}")
            else:
                es_counter += 1
                if es_counter >= ES_PATIENCE_PPO:
                    print(f"\n⏹ PPO early stopping at epoch {epoch}.\n")
                    break

        if best_state is not None:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(OUT_DIR, "ppo_model.bin"))
            print(f"\n✔ Best PPO model saved (dev_rare_f1={best_rare_f1:.4f})")

        ppo_hist_df = pd.DataFrame(history)
        ppo_hist_df.to_csv(os.path.join(OUT_DIR, "ppo_history.csv"), index=False)
        return ppo_hist_df, best_rare_f1


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds,
                               labels=list(range(NUM_LABELS)),
                               average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds,
                                     labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds,
                                  labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_trues  = [id2label[x] for x in all_trues]
    str_preds  = [id2label[x] for x in all_preds]
    cls_report = classification_report(
        str_trues, str_preds, labels=LABELS, digits=4, zero_division=0)
    cm         = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1,
        "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec,
        "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


def count_parameters(model):
    total_trainable = sum(p.numel() for p in model.parameters()
                          if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters()
                          if not p.requires_grad)
    print(f"\n  Trainable: {total_trainable:,} | Frozen: {total_frozen:,}")
    return total_trainable, total_frozen


# ═══════════════════════════════════════════════════════════
# TRAINER  (Phase A: base model)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []
        param_groups.append({
            "params": list(self.model.bert.pooler.parameters()),
            "lr": BERT_LR, "weight_decay": WEIGHT_DECAY,
        })

        encoder_layers = self.model.bert.encoder.layer
        n_layers       = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth  = (n_layers - 1) - i
            lr_i   = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters()
                      if p.requires_grad]
            if params:
                param_groups.append({
                    "params": params, "lr": lr_i, "weight_decay": WEIGHT_DECAY,
                })

        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        param_groups.append({
            "params": head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
        })
        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader     = DataLoader(dataset, batch_size=2, shuffle=False,
                                collate_fn=collate_rrc)
        total_loss = 0.0
        n          = 0
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype,
                                     labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total_loss += loss.item()
                    n          += 1
        return total_loss / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader         = DataLoader(dataset, batch_size=2, shuffle=False,
                                    collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples      = len(dataset)
        infer_start    = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype,
                                        labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents     = len(all_trues)
            infer_info  = {
                "total_inference_time_s":     total_infer,
                "latency_per_document_ms":    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(
                    OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):
        train_loader  = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                    shuffle=True, collate_fn=collate_rrc)
        optimizer     = self.build_optimizer()
        total_steps   = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps  = int(WARMUP_RATIO * total_steps)
        scheduler     = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)

        early_stopper = EarlyStopping()
        history       = []
        best_f1, best_state = -1.0, None
        total_start   = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype,
                                     labels=labels, lengths=lengths)

                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1
                    optimizer.zero_grad()
                    continue

                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                n_steps      += 1

            # Flush any leftover accumulation
            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_loss       = self.compute_val_loss(dev_dataset)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base] Epoch {epoch:03d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | "
                f"ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "base",
                "train_loss": avg_train_loss, "val_loss": val_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_micro_f1": val_metrics["micro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n")
                break

        total_time = time.time() - total_start
        hist_df    = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "base_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")

        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state or self.model.state_dict(),
                       os.path.join(BEST_MODEL_DIR, "base_model.bin"))
            print(f"  Base model saved to {BEST_MODEL_DIR}/base_model.bin")

        return hist_df, total_time


# ═══════════════════════════════════════════════════════════
# KG BUILDER  (Phase A → B)
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_knowledge_graph(base_model, train_docs, tokenizer,
                           device=DEVICE) -> KnowledgeGraph:
    print("\n🔨 Building Knowledge Graph from training embeddings ...")
    base_model.eval()
    base_model.to(device)

    kg     = KnowledgeGraph(emb_dim=base_model.sent_out_dim)
    loader = DataLoader(
        RRCDataset(train_docs, tokenizer),
        batch_size=1, shuffle=False, collate_fn=collate_rrc,
    )

    for doc_idx, (ids, attn, ttype, labels, lengths) in enumerate(loader):
        ids     = ids.to(device)
        attn    = attn.to(device)
        ttype   = ttype.to(device)
        lengths = lengths.to(device)

        sent_vecs = base_model.encode_sentences(ids, attn, ttype).squeeze(0)
        n         = int(lengths[0].item())
        kg.add_nodes(sent_vecs[:n].cpu(), labels[0, :n].tolist())

        if (doc_idx + 1) % 50 == 0:
            print(f"  Processed {doc_idx+1} / {len(loader)} docs")

    kg.build_edges()
    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    kg.save(kg_path)
    return kg


# ═══════════════════════════════════════════════════════════
# TRAINER  (Phase B: KG-augmented fine-tuning)
# ═══════════════════════════════════════════════════════════
class KGTrainer:
    def __init__(self, kg_model: KGAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_params = (list(self.model.gat_fusion.parameters()) +
                      list(self.model.fusion_proj.parameters()) +
                      list(self.model.fusion_classifier.parameters()) +
                      list(self.model.fusion_crf.parameters()))
        base_params = [p for p in self.model.base.parameters()
                       if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_params,   "lr": HEAD_LR,  "weight_decay": WEIGHT_DECAY},
            {"params": base_params,  "lr": BERT_LR,  "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader         = DataLoader(dataset, batch_size=2, shuffle=False,
                                    collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples      = len(dataset)
        infer_start    = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype,
                                        labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents     = len(all_trues)
            infer_info  = {
                "total_inference_time_s": total_infer,
                "latency_per_document_ms":
                    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s":
                    n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(
                    OUT_DIR, f"kg_inference_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              num_epochs=NUM_EPOCHS_KG):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   shuffle=True, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)

        early_stopper       = EarlyStopping(patience=5)
        history             = []
        best_f1, best_state = -1.0, None
        total_start         = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for ids, attn, ttype, labels, lengths in train_loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype,
                                     labels=labels, lengths=lengths)

                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad()
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

                running_loss += loss.item()
                n_steps      += 1

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[KG]   Epoch {epoch:02d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | "
                f"ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "kg",
                "train_loss": avg_train_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best KG val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  KG early stopping at epoch {epoch}.\n")
                break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "kg_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state,
                       os.path.join(BEST_MODEL_DIR, "kg_model.bin"))
            print(f"\n✔ Best KG model saved (val_macro_f1={best_f1:.4f})")

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# PPO FULL EVALUATION
# ═══════════════════════════════════════════════════════════
def evaluate_ppo_full(ppo_trainer: PPOTrainer, dataset,
                      rare_ids: list, split_name: str = "test"):
    macro_f1, rare_f1, all_trues, all_preds = ppo_trainer.evaluate(
        dataset, rare_ids, split_name=split_name)

    acc         = accuracy_score(all_trues, all_preds)
    weighted_f1 = f1_score(all_trues, all_preds,
                           average="weighted", zero_division=0)
    per_cls_f1  = f1_score(all_trues, all_preds,
                           labels=list(range(NUM_LABELS)),
                           average=None, zero_division=0)

    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]
    report    = classification_report(
        str_trues, str_preds, labels=LABELS, digits=4, zero_division=0)

    with open(os.path.join(
            OUT_DIR, f"ppo_{split_name}_classification_report.txt"), "w") as f:
        f.write(f"Model: InLegalBERT + KG-RAG + PPO "
                f"(Phase D: {PHASE_D_MODE})\n\n")
        f.write(report)

    per_class_f1_dict = {id2label[i]: float(per_cls_f1[i])
                         for i in range(NUM_LABELS)}

    return {
        "accuracy": acc, "macro_f1": macro_f1,
        "weighted_f1": weighted_f1, "rare_f1": rare_f1,
        "per_class_f1": per_class_f1_dict,
        "cls_report": report,
        "all_trues": all_trues, "all_preds": all_preds,
    }


# ═══════════════════════════════════════════════════════════
# VISUALISATION HELPERS
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
        for tick in ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name,
                             rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
              for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars    = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved {path}")


def plot_all_history(base_df, kg_df, ppo_df):
    """Combined training curve across all four phases."""
    all_df = pd.concat([base_df, kg_df, ppo_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)

    b1 = len(base_df)
    b2 = b1 + len(kg_df)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"].fillna(method="ffill"),
            label="Train Loss", marker="o", markersize=2)
    ax.axvline(b1, color="orange", linestyle="--", label="Phase B start")
    ax.axvline(b2, color="purple", linestyle="--", label="Phase C start")
    ax.set_title("Combined Training Loss (A→B→C+D)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"].fillna(method="ffill"),
            label="Val Macro-F1", marker="o", markersize=2)
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"].fillna(method="ffill"),
            label="Val Rare-F1", marker="s", markersize=2)
    ax.axvline(b1, color="orange", linestyle="--", label="Phase B start")
    ax.axvline(b2, color="purple", linestyle="--", label="Phase C start")
    ax.set_title("Val F1  (Base → KG-RAG → PPO+RL)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUT_DIR, "all_phases_training_curves.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved {path}")


def print_final_table(kg_dev, kg_test, ppo_dev, ppo_test,
                      base_time, kg_time, ppo_time, total_trainable):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
    ]
    sep = "=" * 80
    print(f"\n{sep}")
    print("FINAL RESULTS  — InLegalBERT + BiLSTM + MHA + CRF + KG-RAG + PPO")
    print(sep)
    print(f"  Trainable parameters : {total_trainable:,}")
    print(f"  Phase A training     : {base_time/60:.1f} min")
    print(f"  Phase B training     : {kg_time/60:.1f} min")
    print(f"  Phase C+D training   : {ppo_time/60:.1f} min")
    print(f"  Phase D mode         : {PHASE_D_MODE}")
    print("-" * 80)
    print(f"  {'Metric':<28} {'KG-Dev':>10} {'KG-Test':>10} "
          f"{'PPO-Dev':>10} {'PPO-Test':>10}")
    print("-" * 80)
    for label, key in rows:
        kd = kg_dev[key]
        kt = kg_test[key]
        pd_val = ppo_dev[key]
        pt_val = ppo_test[key]
        gain_d = pt_val - kd
        gain_t = pt_val - kt
        print(f"  {label:<28} {kd:>10.4f} {kt:>10.4f} "
              f"{pd_val:>10.4f} {pt_val:>10.4f}")
    print(sep)

    print("\n  PER-CLASS F1  (Phase B vs Phase C+D — TEST SET)")
    print("  " + "-" * 70)
    print(f"  {'Label':<22} {'KG-RAG':>10} {'PPO':>10} {'Δ':>8}")
    print("  " + "-" * 70)
    for lbl in LABELS:
        kg_f1  = kg_test["per_class_metrics"][lbl]["f1"]
        ppo_f1 = ppo_test["per_class_f1"].get(lbl, 0.0)
        delta  = ppo_f1 - kg_f1
        flag   = " ↑" if delta > 0.005 else (" ↓" if delta < -0.005 else "")
        print(f"  {lbl:<22} {kg_f1:>10.4f} {ppo_f1:>10.4f} "
              f"{delta:>+8.4f}{flag}")
    print("  " + "-" * 70)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device  : {DEVICE}")
    print("Pipeline: InLegalBERT + BiLSTM + MHA + CRF")
    print("          → KG-RAG (Phase A+B)")
    print(f"          → PPO RL + Synthetic Correction [{PHASE_D_MODE}] (Phase C+D)\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_docs = extract_docs(load_jsonl(TRAIN_PATH))
    dev_docs   = extract_docs(load_jsonl(DEV_PATH))
    test_docs  = extract_docs(load_jsonl(TEST_PATH))
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer ...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ══════════════════════════════════════════════════════
    # PHASE A — Base model training
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training")
    print("=" * 60)

    base_model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name=INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden=SENT_LSTM_HIDDEN, sent_lstm_layers=SENT_LSTM_LAYERS,
        ctx_lstm_hidden=CTX_LSTM_HIDDEN,   ctx_lstm_layers=CTX_LSTM_LAYERS,
        mha_heads=MHA_HEADS, mha_dropout=MHA_DROPOUT,
        num_labels=NUM_LABELS, dropout=DROPOUT,
        freeze_layers=BERT_FREEZE_LAYERS,
    )

    base_trainer = BaseTrainer(base_model, device=DEVICE)
    base_hist_df, base_time = base_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE,
    )

    # ══════════════════════════════════════════════════════
    # PHASE A→B — Build KG
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Building Knowledge Graph")
    print("=" * 60)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if os.path.exists(kg_path):
        print("  Found existing KG, loading ...")
        kg = KnowledgeGraph.load(kg_path, emb_dim=base_model.sent_out_dim)
    else:
        kg = build_knowledge_graph(base_model, train_docs, tokenizer, DEVICE)

    # ══════════════════════════════════════════════════════
    # PHASE B — KG-Augmented fine-tuning
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: KG-Augmented Fine-Tuning")
    print("=" * 60)

    retriever = KGRetriever(kg, top_k=KG_TOP_K,
                            top_nodes=KG_TOP_NODES, hop=KG_HOP)
    kg_model  = KGAugmentedModel(
        base_model=base_model, kg=kg,
        rare_ids=rare_ids, retriever=retriever,
    )

    total_trainable, _ = count_parameters(kg_model)
    kg_trainer         = KGTrainer(kg_model, device=DEVICE)
    kg_hist_df, kg_time = kg_trainer.train(
        train_dataset, dev_dataset, rare_ids, num_epochs=NUM_EPOCHS_KG,
    )

    # ── Phase B evaluation ────────────────────────────────
    print("\nEvaluating Phase B on Dev set ...")
    kg_dev_metrics  = kg_trainer.evaluate(dev_dataset, rare_ids,
                                           split_name="kg_dev",
                                           measure_inference_time=True)
    print(f"  [KG] Dev  Macro-F1: {kg_dev_metrics['macro_f1']:.4f}  "
          f"Rare-F1: {kg_dev_metrics['rare_f1']:.4f}")

    print("\nEvaluating Phase B on Test set ...")
    kg_test_metrics = kg_trainer.evaluate(test_dataset, rare_ids,
                                           split_name="kg_test",
                                           measure_inference_time=True)
    print(f"  [KG] Test Macro-F1: {kg_test_metrics['macro_f1']:.4f}  "
          f"Rare-F1: {kg_test_metrics['rare_f1']:.4f}")

    # Save Phase B reports
    for name, metrics in [("kg_dev", kg_dev_metrics),
                           ("kg_test", kg_test_metrics)]:
        with open(os.path.join(OUT_DIR,
                  f"{name}_classification_report.txt"), "w") as f:
            f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + KG-RAG\n\n")
            f.write(metrics["cls_report"])
        save_confusion_matrix(metrics["cm"], name, rare_labels)
        save_per_class_f1_chart(metrics["per_class_metrics"],
                                name, rare_labels)

    # ══════════════════════════════════════════════════════
    # PHASE C+D — PPO RL + Synthetic Correction
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE C+D: PPO RL + Synthetic Correction")
    print("=" * 60)

    ppo_model   = PPOModel(kg_model=kg_model, rare_ids=rare_ids)
    ppo_trainer = PPOTrainer(
        ppo_model=ppo_model, rare_ids=rare_ids,
        kg_retriever=retriever, device=DEVICE,
    )

    ppo_start = time.time()
    ppo_hist_df, best_rare_f1 = ppo_trainer.train(
        train_dataset, dev_dataset, rare_ids, num_epochs=PPO_EPOCHS,
    )
    ppo_time = time.time() - ppo_start

    # ── Phase C+D evaluation ──────────────────────────────
    print("\nEvaluating Phase C+D on Dev set ...")
    ppo_dev_metrics  = evaluate_ppo_full(
        ppo_trainer, dev_dataset, rare_ids, "ppo_dev")
    print(f"  [PPO] Dev  Macro-F1: {ppo_dev_metrics['macro_f1']:.4f}  "
          f"Rare-F1: {ppo_dev_metrics['rare_f1']:.4f}")

    print("\nEvaluating Phase C+D on Test set ...")
    ppo_test_metrics = evaluate_ppo_full(
        ppo_trainer, test_dataset, rare_ids, "ppo_test")
    print(f"  [PPO] Test Macro-F1: {ppo_test_metrics['macro_f1']:.4f}  "
          f"Rare-F1: {ppo_test_metrics['rare_f1']:.4f}")

    # Save PPO confusion matrices + F1 charts
    ppo_cm_trues = ppo_test_metrics["all_trues"]
    ppo_cm_preds = ppo_test_metrics["all_preds"]
    ppo_cm       = confusion_matrix(
        [id2label[x] for x in ppo_cm_trues],
        [id2label[x] for x in ppo_cm_preds],
        labels=LABELS,
    )
    save_confusion_matrix(ppo_cm, "ppo_test", rare_labels)

    # Build per_class_metrics structure for chart
    ppo_pcm = {
        lbl: {"f1": ppo_test_metrics["per_class_f1"].get(lbl, 0.0),
              "precision": 0.0, "recall": 0.0}
        for lbl in LABELS
    }
    save_per_class_f1_chart(ppo_pcm, "ppo_test", rare_labels)

    # Prediction CSV
    pd.DataFrame({
        "true": [id2label[x] for x in ppo_cm_trues],
        "pred": [id2label[x] for x in ppo_cm_preds],
    }).to_csv(os.path.join(OUT_DIR, "ppo_test_predictions.csv"), index=False)

    # ── Combined training curves ───────────────────────────
    ppo_hist_rename = ppo_hist_df.rename(columns={
        "dev_macro_f1": "val_macro_f1",
        "dev_rare_f1":  "val_rare_f1",
        "pol_loss":     "train_loss",
    })
    plot_all_history(base_hist_df, kg_hist_df, ppo_hist_rename)

    # ── Final JSON summary ─────────────────────────────────
    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision",
        "macro_recall", "micro_recall", "weighted_recall",
        "rare_precision", "rare_recall", "accuracy",
    ]
    summary = {
        "model": "InLegalBERT + BiLSTM + MHA + CRF + KG-RAG + PPO",
        "phase_d_mode": PHASE_D_MODE,
        "rare_classes": rare_labels,
        "timing": {
            "phase_a_s": base_time,
            "phase_b_s": kg_time,
            "phase_cd_s": ppo_time,
            "total_s": base_time + kg_time + ppo_time,
        },
        "ppo_hyperparams": {
            "reward_rare_correct": REWARD_RARE_CORRECT,
            "reward_rare_wrong":   REWARD_RARE_WRONG,
            "reward_maj_correct":  REWARD_MAJ_CORRECT,
            "ppo_clip_eps":        PPO_CLIP_EPS,
            "ppo_kl_coef":         PPO_KL_COEF,
            "gae_gamma":           GAE_GAMMA,
            "gae_lambda":          GAE_LAMBDA,
            "rollout_docs":        ROLLOUT_DOCS,
        },
        "kg_rag_dev":   {k: kg_dev_metrics[k]  for k in scalar_keys
                         if k in kg_dev_metrics},
        "kg_rag_test":  {k: kg_test_metrics[k] for k in scalar_keys
                         if k in kg_test_metrics},
        "ppo_dev":  {k: ppo_dev_metrics[k]  for k in scalar_keys
                     if k in ppo_dev_metrics},
        "ppo_test": {k: ppo_test_metrics[k] for k in scalar_keys
                     if k in ppo_test_metrics},
        "per_class_kg_test":  kg_test_metrics["per_class_metrics"],
        "per_class_ppo_test": ppo_test_metrics["per_class_f1"],
    }
    with open(os.path.join(OUT_DIR, "full_metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # ── Console table ──────────────────────────────────────
    print_final_table(
        kg_dev_metrics, kg_test_metrics,
        ppo_dev_metrics, ppo_test_metrics,
        base_time, kg_time, ppo_time, total_trainable,
    )

    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device  : cuda:0
Pipeline: InLegalBERT + BiLSTM + MHA + CRF
          → KG-RAG (Phase A+B)
          → PPO RL + Synthetic Correction [combined] (Phase C+D)

Loading JSONL files ...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RARE
   NONE                  4.79%  ( 1377 samples) ← RARE

   Rare classes (10): ['RLC', 'ISSUE', 'ARG_

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-7.
🔥 BERT layers trainable: layers 8-11 + pooler.

[Base] Epoch 001/60 | train_loss: 287.7585 | val_loss: 211.9501 | val_macro_f1: 0.0391 | val_rare_f1: 0.0000 | time: 42.7s | ES: 0/10
  ✔ New best val_macro_f1=0.0391
[Base] Epoch 002/60 | train_loss: 234.5406 | val_loss: 175.5814 | val_macro_f1: 0.0797 | val_rare_f1: 0.0000 | time: 37.6s | ES: 0/10
  ✔ New best val_macro_f1=0.0797
[Base] Epoch 003/60 | train_loss: 198.7177 | val_loss: 128.8831 | val_macro_f1: 0.2005 | val_rare_f1: 0.0560 | time: 38.2s | ES: 0/10
  ✔ New best val_macro_f1=0.2005
[Base] Epoch 004/60 | train_loss: 166.5206 | val_loss: 106.5727 | val_macro_f1: 0.2568 | val_rare_f1: 0.0970 | time: 38.0s | ES: 0/10
  ✔ New best val_macro_f1=0.2568
[Base] Epoch 005/60 | train_loss: 140.3391 | val_loss: 88.4431 | val_macro_f1: 0.2828 | val_rare_f1: 0.1195 | time: 37.7s | ES: 0/10
  ✔ New best val_macro_f1=0.2828
[Base] Epoch 006/60 | train_loss: 124.8879 | val_loss: 83.0902 | val

RuntimeError: size mismatch, got input (4167), mat (4167x256), vec (128)

In [2]:
# resume_ppo_phase_cd.py
#
# Resumes from Phase C+D (PPO) using saved Phase B kg_model.bin
# FIX: KG retrieval uses sent_vecs (dim=256) not ctx_out (dim=128)
#
# Place this file alongside your original script.
# It loads the already-trained base_model.bin + kg_model.bin,
# skips Phase A & B, and runs Phase C+D directly.

import os, json, random, time, math
from collections import Counter, defaultdict, deque
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)


# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH     = "dataset/build_train.jsonl"
DEV_PATH       = "dataset/build_dev.jsonl"
TEST_PATH      = "dataset/build_test.jsonl"
OUT_DIR        = "rrc_kg_rag_ppo_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS = 8
BERT_LR_DECAY      = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

AUX_CE_WEIGHT   = 0.2
LABEL_SMOOTHING = 0.1
ES_MIN_DELTA    = 1e-4
WARMUP_RATIO    = 0.05
RARE_THRESHOLD  = 0.05

KG_TOP_K           = 3
KG_TOP_NODES       = 5
KG_HOP             = 1
UNCERTAINTY_THRESH = 0.7
RARE_ALWAYS_KG     = True
KG_FUSION_DIM      = 256

RST_INTRA_THRESH = 0.6
RST_CROSS_THRESH = 0.5

# PPO / Phase D
REWARD_RARE_CORRECT =  2.0
REWARD_RARE_WRONG   = -3.0
REWARD_MAJ_CORRECT  =  0.3
REWARD_MAJ_WRONG    =  0.0

PPO_EPOCHS       = 20
PPO_MINI_EPOCHS  = 4
PPO_CLIP_EPS     = 0.2
PPO_KL_COEF      = 0.1
PPO_VALUE_COEF   = 0.5
PPO_ENTROPY_COEF = 0.01
PPO_LR           = 3e-5
PPO_GRAD_CLIP    = 1.0
GAE_GAMMA        = 0.99
GAE_LAMBDA       = 0.95
ROLLOUT_DOCS     = 16
ES_PATIENCE_PPO  = 7

PHASE_D_MODE = "combined"

CORRECTION_LR        = 1e-4
CORRECTION_EPOCHS    = 15
CORRECTION_NEG_RATIO = 3

CURRICULUM_WINDOW = 5
CURRICULUM_MIN_W  = 0.5
CURRICULUM_MAX_W  = 3.0
KG_DISAGREE_BONUS = 0.8

# ═══════════════════════════════════════════════════════════
# LABELS
# ═══════════════════════════════════════════════════════════
LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# DATA
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        all_docs.append((sents[:max_sents], labs[:max_sents]))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    return rare_labels, rare_ids, label_freqs


class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents, padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get("token_type_ids",
                                      torch.zeros_like(enc["input_ids"])),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MODEL ARCHITECTURE (identical to original)
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query      = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            attn = attn.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        attn = self.attn_drop(F.softmax(attn, dim=-1))
        return self.out_proj(torch.matmul(attn, V).squeeze(2).reshape(N, H))


class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):
    def __init__(self, bert_model_name=INLEGALBERT_MODEL_NAME,
                 sent_lstm_hidden=SENT_LSTM_HIDDEN, sent_lstm_layers=SENT_LSTM_LAYERS,
                 ctx_lstm_hidden=CTX_LSTM_HIDDEN,   ctx_lstm_layers=CTX_LSTM_LAYERS,
                 mha_heads=MHA_HEADS, mha_dropout=MHA_DROPOUT,
                 num_labels=NUM_LABELS, dropout=DROPOUT,
                 freeze_layers=BERT_FREEZE_LAYERS):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0)
        self.sent_out_dim = sent_lstm_hidden * 2  # 256

        self.mha_pooling     = MultiHeadAttentionPooling(self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=self.sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0)
        self.ctx_out_dim = ctx_lstm_hidden * 2  # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(self.ctx_out_dim // 2, num_labels))
        self.crf     = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

    def _freeze_bert_layers(self, n_freeze):
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i in range(min(n_freeze, len(self.bert.encoder.layer))):
            for p in self.bert.encoder.layer[i].parameters():
                p.requires_grad = False

    def encode_sentences(self, input_ids, attention_mask, token_type_ids, lengths=None):
        B, T, L = input_ids.shape
        N       = B * T
        flat_ids, flat_mask, flat_types = (
            input_ids.view(N, L), attention_mask.view(N, L), token_type_ids.view(N, L))
        valid = flat_mask.sum(dim=-1) > 0
        token_embs = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(input_ids=flat_ids[valid], attention_mask=flat_mask[valid],
                            token_type_ids=flat_types[valid])
            token_embs[valid] = out.last_hidden_state.to(token_embs.dtype)
        token_embs = self.dropout(token_embs)
        lstm_out, _ = self.sent_bilstm(token_embs)
        lstm_out    = self.dropout(lstm_out)
        pad_mask    = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs   = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs   = self.sent_layer_norm(sent_vecs)
        sent_vecs   = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def get_emissions(self, input_ids, attention_mask, token_type_ids, lengths=None):
        sent_vecs = self.encode_sentences(input_ids, attention_mask, token_type_ids)
        drop      = self.dropout(sent_vecs)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(drop)
        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def forward(self, input_ids, attention_mask, token_type_ids, labels=None, lengths=None):
        _, _, emissions = self.get_emissions(input_ids, attention_mask, token_type_ids, lengths)
        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool, device=emissions.device)
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            crf_loss = -self.crf(emissions, safe, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(emissions.reshape(B2*T2, C), labels.reshape(B2*T2))
            return crf_loss + AUX_CE_WEIGHT * ce_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# KNOWLEDGE GRAPH
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    def __init__(self, emb_dim=KG_FUSION_DIM):
        self.emb_dim     = emb_dim
        self.nodes       = defaultdict(list)
        self.intra_edges = defaultdict(list)
        self.cross_edges = []
        self._stacked    = {}

    def _get_stacked(self, lid):
        if lid not in self._stacked:
            if lid not in self.nodes or not self.nodes[lid]:
                return None
            self._stacked[lid] = torch.stack([n["emb"] for n in self.nodes[lid]])
        return self._stacked[lid]

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM):
        kg = cls(emb_dim=emb_dim)
        with open(path) as f:
            data = json.load(f)
        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                kg.nodes[lid].append({"emb": torch.tensor(n["emb"]), "text": n["text"]})
        for k, edges in data["intra_edges"].items():
            kg.intra_edges[int(k)] = [tuple(e) for e in edges]
        kg.cross_edges = [tuple(e) for e in data["cross_edges"]]
        print(f"  KG loaded from {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS, threshold=UNCERTAINTY_THRESH):
        self.log_C     = math.log(num_classes)
        self.threshold = threshold

    def entropy(self, logits):
        probs = F.softmax(logits, dim=-1)
        return -(probs * (probs + 1e-9).log()).sum(dim=-1) / self.log_C

    def is_uncertain(self, logits):
        return self.entropy(logits) > self.threshold

    def top_label(self, logits):
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# KG RETRIEVER
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    def __init__(self, kg: KnowledgeGraph, top_k=KG_TOP_K,
                 top_nodes=KG_TOP_NODES, hop=KG_HOP):
        self.kg        = kg
        self.top_k     = top_k
        self.top_nodes = top_nodes
        self.hop       = hop

    def retrieve(self, h_i: torch.Tensor, rare_ids=None, first_pass_label=None):
        # h_i must be dim=256 (sent_out_dim) to match KG embeddings
        h_norm = F.normalize(h_i.unsqueeze(0), dim=-1)
        subgraph_scores = {}
        for lid in self.kg.nodes:
            embs = self.kg._get_stacked(lid)
            if embs is None or embs.shape[0] == 0:
                continue
            sims = torch.mv(F.normalize(embs, dim=-1), h_norm.squeeze(0))
            subgraph_scores[lid] = float(sims.max().item())

        sorted_sgs = sorted(subgraph_scores.items(), key=lambda x: -x[1])
        selected   = [lid for lid, _ in sorted_sgs[:self.top_k]]
        results    = []

        for lid in selected:
            embs = self.kg._get_stacked(lid)
            if embs is None:
                continue
            sims    = torch.mv(F.normalize(embs, dim=-1), h_norm.squeeze(0))
            k       = min(self.top_nodes, embs.shape[0])
            top_idx = sims.topk(k).indices.tolist()
            top_sim = sims.topk(k).values.tolist()
            seed_set = set(top_idx)

            if self.hop >= 1:
                for idx in list(seed_set):
                    for (i, j, w) in self.kg.intra_edges.get(lid, []):
                        if i == idx and j not in seed_set:
                            seed_set.add(j)
                        elif j == idx and i not in seed_set:
                            seed_set.add(i)
                for (la, ni, lb, nj, w) in self.kg.cross_edges:
                    if la == lid and ni in seed_set:
                        ce = self.kg._get_stacked(lb)
                        if ce is not None and nj < ce.shape[0]:
                            results.append((ce[nj], w))
                    elif lb == lid and nj in seed_set:
                        ce = self.kg._get_stacked(la)
                        if ce is not None and ni < ce.shape[0]:
                            results.append((ce[ni], w))

            for idx, sim in zip(top_idx, top_sim):
                results.append((embs[idx], float(sim)))
        return results


# ═══════════════════════════════════════════════════════════
# GRAPH ATTENTION FUSION
# ═══════════════════════════════════════════════════════════
class GraphAttentionFusion(nn.Module):
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT):
        super().__init__()
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, h_i, neighbours, device=None):
        if not neighbours:
            return torch.zeros_like(h_i)
        if device is None:
            device = h_i.device
        embs    = torch.stack([nb[0] for nb in neighbours]).to(device)
        weights = torch.tensor([nb[1] for nb in neighbours], device=device, dtype=torch.float)
        q   = self.proj_q(h_i.unsqueeze(0))
        k   = self.proj_k(embs)
        dot = torch.mv(k, q.squeeze(0)) * self.scale
        alpha = self.dropout(F.softmax(dot * weights, dim=0))
        return (alpha.unsqueeze(-1) * embs).sum(dim=0)


# ═══════════════════════════════════════════════════════════
# KG-AUGMENTED MODEL
# ═══════════════════════════════════════════════════════════
class KGAugmentedModel(nn.Module):
    def __init__(self, base_model, kg, rare_ids, retriever=None):
        super().__init__()
        self.base        = base_model
        self.kg          = kg
        self.rare_ids    = rare_ids
        self.retriever   = retriever or KGRetriever(kg)
        self.uncertainty = UncertaintyEstimator()

        sent_dim = base_model.sent_out_dim  # 256
        ctx_dim  = base_model.ctx_out_dim   # 128

        self.gat_fusion  = GraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_proj = nn.Sequential(
            nn.Linear(sent_dim * 2, ctx_dim), nn.GELU(), nn.Dropout(DROPOUT))
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT), nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(ctx_dim // 2, NUM_LABELS))
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

    def _kg_fuse_batch(self, sent_vecs, emissions, lengths, device):
        B, T, sent_dim = sent_vecs.shape
        fused          = sent_vecs.clone()
        uncertain_mask = self.uncertainty.is_uncertain(emissions)
        top_labels     = self.uncertainty.top_label(emissions)

        for b in range(B):
            n = int(lengths[b].item())
            for t in range(n):
                uncertain = bool(uncertain_mask[b, t].item())
                pred_lbl  = int(top_labels[b, t].item())
                if not (uncertain or (RARE_ALWAYS_KG and pred_lbl in self.rare_ids)):
                    continue
                h_i        = sent_vecs[b, t].detach().cpu()
                neighbours = self.retriever.retrieve(h_i, rare_ids=self.rare_ids,
                                                     first_pass_label=pred_lbl)
                if not neighbours:
                    continue
                v_i = self.gat_fusion(sent_vecs[b, t], neighbours, device=device)
                fused[b, t] = sent_vecs[b, t] + v_i
        return fused

    def forward(self, input_ids, attention_mask, token_type_ids, labels=None, lengths=None):
        device = input_ids.device
        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)
        fused_sent  = self._kg_fuse_batch(sent_vecs, base_emissions, lengths, device)
        fused_drop  = self.base.dropout(fused_sent)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)

        fused_ctx       = self.base.dropout(fused_ctx)
        fused_emissions = torch.nan_to_num(
            self.fusion_classifier(fused_ctx), nan=0.0, posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2], dtype=torch.bool, device=device)

        combined = (base_emissions + fused_emissions) / 2.0

        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            loss = (
                -self.base.crf(base_emissions, safe, mask=mask, reduction="mean")
                - self.fusion_crf(fused_emissions, safe, mask=mask, reduction="mean")
            ) / 2.0
            B2, T2, C = combined.shape
            loss += AUX_CE_WEIGHT * self.ce_loss(
                combined.reshape(B2*T2, C), labels.reshape(B2*T2))
            return loss, combined
        else:
            return self.fusion_crf.decode(fused_emissions, mask=mask), fused_emissions


# ═══════════════════════════════════════════════════════════
# PPO COMPONENTS
# ═══════════════════════════════════════════════════════════
class ActorHead(nn.Module):
    def __init__(self, ctx_dim, num_labels=NUM_LABELS, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(ctx_dim // 2, num_labels))

    def forward(self, ctx_out):
        return self.net(ctx_out)

    def copy_from_fusion_classifier(self, fusion_classifier):
        try:
            self.net.load_state_dict(fusion_classifier.state_dict())
            print("  ✔ ActorHead warm-started from Phase B fusion_classifier.")
        except Exception as e:
            print(f"  ⚠ ActorHead warm-start failed ({e}), using random init.")


class CriticHead(nn.Module):
    def __init__(self, ctx_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(), nn.Linear(ctx_dim // 2, 1))

    def forward(self, ctx_out):
        return self.net(ctx_out).squeeze(-1)


class PPOModel(nn.Module):
    def __init__(self, kg_model, rare_ids):
        super().__init__()
        self.kg_model = kg_model
        self.rare_ids = set(rare_ids)
        ctx_dim       = kg_model.base.ctx_out_dim  # 128
        self.actor    = ActorHead(ctx_dim)
        self.critic   = CriticHead(ctx_dim)
        self.actor.copy_from_fusion_classifier(kg_model.fusion_classifier)

    def get_ctx_out(self, input_ids, attention_mask, token_type_ids, lengths):
        """
        Returns (sent_vecs [dim=256], fused_ctx [dim=128], base_emissions).
        sent_vecs is exposed so KG retrieval always uses the correct dim.
        """
        kg   = self.kg_model
        base = kg.base

        sent_vecs, _, base_emissions = base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)
        fused_sent = kg._kg_fuse_batch(
            sent_vecs, base_emissions, lengths, input_ids.device)

        fused_drop = base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = base.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            ctx_out, _ = base.ctx_bilstm(fused_drop)

        # Return BOTH sent_vecs (256-d, for KG retrieval) and ctx_out (128-d, for actor/critic)
        return fused_sent, base.dropout(ctx_out), base_emissions

    @torch.no_grad()
    def decode_actions(self, actor_logits, lengths):
        B, T, _ = actor_logits.shape
        mask    = torch.zeros(B, T, dtype=torch.bool, device=actor_logits.device)
        for i, l in enumerate(lengths):
            mask[i, :l] = True
        return self.kg_model.fusion_crf.decode(actor_logits, mask=mask)


class RewardShaper:
    def __init__(self, rare_ids):
        self.rare_ids = set(rare_ids)
    def base_reward(self, pred, true):
        correct = (pred == true)
        if true in self.rare_ids:
            return REWARD_RARE_CORRECT if correct else REWARD_RARE_WRONG
        else:
            return REWARD_MAJ_CORRECT if correct else REWARD_MAJ_WRONG


def compute_gae(rewards, values, gamma=GAE_GAMMA, lam=GAE_LAMBDA):
    T = len(rewards); advantages = [0.0]*T; returns = [0.0]*T; gae = 0.0
    for t in reversed(range(T)):
        nv    = values[t+1] if t+1 < T else 0.0
        delta = rewards[t] + gamma*nv - values[t]
        gae   = delta + gamma*lam*gae
        advantages[t] = gae; returns[t] = gae + values[t]
    return advantages, returns


class RolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states = []; self.actor_logits = []; self.actions = []
        self.true_labels = []; self.advantages = []; self.returns = []
        self.old_log_probs = []

    def add_trajectory(self, states_t, logits_t, actions_t,
                       trues_t, advantages_t, returns_t):
        for i in range(len(actions_t)):
            self.states.append(states_t[i])
            self.actor_logits.append(logits_t[i])
            self.actions.append(int(actions_t[i]))
            self.true_labels.append(int(trues_t[i]))
            self.advantages.append(float(advantages_t[i]))
            self.returns.append(float(returns_t[i]))
            lp = F.log_softmax(logits_t[i], dim=-1)[int(actions_t[i])]
            self.old_log_probs.append(float(lp.item()))

    def __len__(self):
        return len(self.actions)

    def as_tensors(self, device):
        states        = torch.stack(self.states).to(device)
        actor_logits  = torch.stack(self.actor_logits).to(device)
        actions       = torch.tensor(self.actions,       dtype=torch.long,  device=device)
        advantages    = torch.tensor(self.advantages,    dtype=torch.float, device=device)
        returns       = torch.tensor(self.returns,       dtype=torch.float, device=device)
        old_log_probs = torch.tensor(self.old_log_probs, dtype=torch.float, device=device)
        adv_std = advantages.std() + 1e-8
        advantages = (advantages - advantages.mean()) / adv_std
        return states, actor_logits, actions, advantages, returns, old_log_probs


# ═══════════════════════════════════════════════════════════
# PHASE D COMPONENTS
# ═══════════════════════════════════════════════════════════
class SyntheticCorrectionModel(nn.Module):
    def __init__(self, ctx_dim, num_labels=NUM_LABELS, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(ctx_dim + num_labels*2, hidden), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden//2), nn.GELU(),
            nn.Linear(hidden//2, 1), nn.Tanh())

    def forward(self, ctx_emb, pred_ids, true_ids):
        C = NUM_LABELS
        x = torch.cat([ctx_emb,
                        F.one_hot(pred_ids, C).float(),
                        F.one_hot(true_ids, C).float()], dim=-1)
        return self.net(x).squeeze(-1)


def collect_correction_data(ppo_model, train_dataset, rare_ids, device=DEVICE):
    ppo_model.eval()
    loader   = DataLoader(train_dataset, batch_size=1, shuffle=False, collate_fn=collate_rrc)
    rare_set = set(rare_ids)
    data     = []
    with torch.no_grad():
        for ids, attn, ttype, labels, lengths in loader:
            ids = ids.to(device); attn = attn.to(device)
            ttype = ttype.to(device); lengths = lengths.to(device)
            # get ctx_out (128-d) for correction model input
            _, ctx_out, _ = ppo_model.get_ctx_out(ids, attn, ttype, lengths)
            actor_logits  = ppo_model.actor(ctx_out)
            probs         = F.softmax(actor_logits, dim=-1)
            n = int(lengths[0].item())
            for t in range(n):
                true_lbl = int(labels[0, t].item())
                if true_lbl < 0 or true_lbl not in rare_set:
                    continue
                pred_lbl   = int(actor_logits[0, t].argmax().item())
                confidence = float(probs[0, t].max().item())
                ctx_emb    = ctx_out[0, t].cpu()
                if pred_lbl != true_lbl:
                    data.append({"ctx_emb": ctx_emb, "pred": pred_lbl,
                                 "true": true_lbl, "target": -1.0})
                elif confidence >= 0.7:
                    data.append({"ctx_emb": ctx_emb, "pred": pred_lbl,
                                 "true": true_lbl, "target": +1.0})
    neg = [d for d in data if d["target"] < 0]
    pos = [d for d in data if d["target"] > 0]
    print(f"  Correction data: {len(neg)} errors, {len(pos)} confident-correct")
    return data


def train_correction_model(corr_model, correction_data, device=DEVICE):
    if not correction_data:
        print("  ⚠ No correction data."); return corr_model
    optimizer = torch.optim.AdamW(corr_model.parameters(), lr=CORRECTION_LR, weight_decay=0.01)
    corr_model.to(device).train()
    neg_data = [d for d in correction_data if d["target"] < 0]
    pos_data = [d for d in correction_data if d["target"] > 0]
    for epoch in range(CORRECTION_EPOCHS):
        if pos_data:
            n_pos  = min(len(pos_data), 64)
            n_neg  = min(len(neg_data), n_pos * CORRECTION_NEG_RATIO)
            batch  = random.sample(pos_data, n_pos) + random.sample(neg_data, n_neg)
        else:
            batch = random.sample(neg_data, min(len(neg_data), 256))
        random.shuffle(batch)
        ctx_embs = torch.stack([b["ctx_emb"] for b in batch]).to(device)
        preds    = torch.tensor([b["pred"]   for b in batch], dtype=torch.long,  device=device)
        trues    = torch.tensor([b["true"]   for b in batch], dtype=torch.long,  device=device)
        targets  = torch.tensor([b["target"] for b in batch], dtype=torch.float, device=device)
        loss = F.mse_loss(corr_model(ctx_embs, preds, trues), targets)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(corr_model.parameters(), 1.0)
        optimizer.step()
        if (epoch+1) % 5 == 0:
            print(f"    [Correction] epoch {epoch+1}/{CORRECTION_EPOCHS} loss={loss.item():.4f}")
    torch.save(corr_model.state_dict(), os.path.join(OUT_DIR, "correction_model.bin"))
    print("  ✔ Correction model saved.")
    return corr_model


def kg_disagree_corrections(kg_retriever, sent_vecs_cpu, actor_preds,
                              rare_ids, bonus=KG_DISAGREE_BONUS):
    """
    FIX: accepts sent_vecs (dim=256) not ctx_out (dim=128).
    KG nodes were built from sent_out_dim=256 embeddings.
    """
    rare_set    = set(rare_ids)
    corrections = []
    for h_i, pred in zip(sent_vecs_cpu, actor_preds):
        # h_i is dim=256 — matches KG embedding dimension
        neighbours = kg_retriever.retrieve(h_i, rare_ids=rare_ids)
        if not neighbours:
            corrections.append(0.0); continue
        total_sim = sum(max(w, 0.0) for _, w in neighbours)
        corrections.append(bonus if (pred not in rare_set and total_sim > 0.5) else 0.0)
    return corrections


class CurriculumScheduler:
    def __init__(self, rare_ids, num_labels=NUM_LABELS,
                 window=CURRICULUM_WINDOW, min_w=CURRICULUM_MIN_W, max_w=CURRICULUM_MAX_W):
        self.rare_ids    = set(rare_ids)
        self.num_labels  = num_labels
        self.window      = window; self.min_w = min_w; self.max_w = max_w
        self.f1_history  = {i: deque(maxlen=window) for i in range(num_labels)}
        self.multipliers = {i: max_w if i in self.rare_ids else 1.0
                            for i in range(num_labels)}

    def update(self, all_trues, all_preds):
        per_class_f1 = f1_score(all_trues, all_preds,
                                 labels=list(range(self.num_labels)),
                                 average=None, zero_division=0)
        for i in range(self.num_labels):
            self.f1_history[i].append(float(per_class_f1[i]))
        for i in self.rare_ids:
            hist = list(self.f1_history[i])
            if len(hist) < 2: continue
            mean_f1  = np.mean(hist); trend = hist[-1] - hist[0]
            target_w = self.max_w * (1.0 - mean_f1) + self.min_w * mean_f1
            if trend > 0.01: target_w *= 0.9
            self.multipliers[i] = float(np.clip(
                0.7*self.multipliers[i] + 0.3*target_w, self.min_w, self.max_w))

    def get_multiplier(self, label_id):
        return self.multipliers.get(label_id, 1.0)

    def log(self):
        rare_mults = {id2label[i]: f"{self.multipliers[i]:.2f}"
                      for i in sorted(self.rare_ids) if i in self.multipliers}
        print(f"  Curriculum multipliers: {rare_mults}")


def compute_shaped_reward(pred, true, rare_ids, base_shaper,
                           correction_model=None, ctx_emb_cpu=None,
                           curriculum=None, kg_delta=0.0, device=DEVICE):
    r = base_shaper.base_reward(pred, true)
    if curriculum is not None and true in rare_ids:
        r *= curriculum.get_multiplier(true)
    r += kg_delta
    if correction_model is not None and ctx_emb_cpu is not None:
        with torch.no_grad():
            ce = ctx_emb_cpu.unsqueeze(0).to(device)
            pd = torch.tensor([pred], dtype=torch.long, device=device)
            td = torch.tensor([true], dtype=torch.long, device=device)
            r += float(correction_model(ce, pd, td).item())
    return r


# ═══════════════════════════════════════════════════════════
# PPO TRAINER  ← KEY FIX HERE
# ═══════════════════════════════════════════════════════════
class PPOTrainer:
    def __init__(self, ppo_model, rare_ids, kg_retriever=None, device=DEVICE):
        self.model       = ppo_model.to(device)
        self.device      = device
        self.rare_ids    = set(rare_ids)
        self.retriever   = kg_retriever
        self.base_shaper = RewardShaper(list(rare_ids))
        self.correction_model = None
        self.curriculum       = None

    def build_optimizer(self):
        actor_params = (list(self.model.actor.parameters()) +
                        list(self.model.critic.parameters()))
        base_params  = [p for p in self.model.kg_model.parameters() if p.requires_grad]
        return torch.optim.AdamW([
            {"params": actor_params, "lr": PPO_LR,       "weight_decay": 0.01},
            {"params": base_params,  "lr": PPO_LR * 0.1, "weight_decay": 0.01},
        ])

    @torch.no_grad()
    def collect_rollout(self, loader_iter, loader, n_docs=ROLLOUT_DOCS):
        self.model.eval()
        buffer = RolloutBuffer(); collected = 0

        while collected < n_docs:
            try:
                batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(loader)
                batch = next(loader_iter)

            ids, attn, ttype, labels, lengths = batch
            ids = ids.to(self.device); attn = attn.to(self.device)
            ttype = ttype.to(self.device); lengths = lengths.to(self.device)

            # ── FIX: get_ctx_out now returns (sent_vecs, ctx_out, base_emissions) ──
            sent_vecs, ctx_out, _ = self.model.get_ctx_out(ids, attn, ttype, lengths)
            actor_logits = self.model.actor(ctx_out)
            values       = self.model.critic(ctx_out)
            decoded      = self.model.decode_actions(actor_logits, lengths)

            for b in range(ids.shape[0]):
                n         = int(lengths[b].item())
                actions_b = decoded[b][:n]
                trues_b   = labels[b, :n].tolist()
                if any(t < 0 for t in trues_b):
                    continue

                # KG disagreement: use sent_vecs (dim=256) — CORRECT dimension
                if (self.retriever is not None and
                        PHASE_D_MODE in ("kg_disagree", "combined")):
                    sent_cpu = [sent_vecs[b, t].detach().cpu() for t in range(n)]
                    kg_delts = kg_disagree_corrections(
                        self.retriever, sent_cpu, actions_b, list(self.rare_ids))
                else:
                    kg_delts = [0.0] * n

                rewards_b = []
                for t in range(n):
                    r = compute_shaped_reward(
                        pred=actions_b[t], true=trues_b[t],
                        rare_ids=self.rare_ids, base_shaper=self.base_shaper,
                        correction_model=(self.correction_model
                                          if PHASE_D_MODE in ("self_play", "combined") else None),
                        # correction model uses ctx_out (dim=128) — correct
                        ctx_emb_cpu=ctx_out[b, t].detach().cpu(),
                        curriculum=(self.curriculum
                                    if PHASE_D_MODE in ("curriculum", "combined") else None),
                        kg_delta=kg_delts[t], device=self.device)
                    rewards_b.append(r)

                vals_b = values[b, :n].detach().cpu().tolist()
                adv_b, ret_b = compute_gae(rewards_b, vals_b)

                buffer.add_trajectory(
                    states_t    =[ctx_out[b, t].detach() for t in range(n)],
                    logits_t    =[actor_logits[b, t].detach() for t in range(n)],
                    actions_t   =actions_b,
                    trues_t     =trues_b,
                    advantages_t=adv_b,
                    returns_t   =ret_b)
                collected += 1

        return buffer, loader_iter

    def ppo_update(self, buffer):
        self.model.train()
        optimizer = self._optimizer
        (states, old_logits, actions, advantages,
         returns, old_log_probs) = buffer.as_tensors(self.device)

        N = len(actions); indices = torch.randperm(N)
        total_pol, total_val, n_up = 0.0, 0.0, 0

        for _ in range(PPO_MINI_EPOCHS):
            for start in range(0, N, 256):
                idx = indices[start:start+256]
                if len(idx) == 0: continue
                s=states[idx]; a=actions[idx]; adv=advantages[idx]
                ret=returns[idx]; olp=old_log_probs[idx]; oleg=old_logits[idx]

                new_logits    = self.model.actor.net(s)
                new_vals      = self.model.critic.net(s).squeeze(-1)
                new_lp        = F.log_softmax(new_logits, dim=-1).gather(1, a.unsqueeze(1)).squeeze(1)
                ratio         = (new_lp - olp).exp()
                clipped       = torch.clamp(ratio, 1-PPO_CLIP_EPS, 1+PPO_CLIP_EPS)
                pol_loss      = -torch.min(ratio*adv, clipped*adv).mean()
                val_loss      = F.mse_loss(new_vals, ret)
                entropy       = -(F.softmax(new_logits, dim=-1) *
                                  F.log_softmax(new_logits, dim=-1)).sum(-1).mean()
                kl_loss       = F.kl_div(F.log_softmax(new_logits, dim=-1),
                                         F.softmax(oleg, dim=-1), reduction="batchmean")
                loss = (pol_loss + PPO_VALUE_COEF*val_loss
                        - PPO_ENTROPY_COEF*entropy + PPO_KL_COEF*kl_loss)

                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), PPO_GRAD_CLIP)
                optimizer.step()
                total_pol += pol_loss.item(); total_val += val_loss.item(); n_up += 1

        return total_pol / max(1, n_up), total_val / max(1, n_up)

    def evaluate(self, dataset, rare_ids, split_name="dev"):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); lengths = lengths.to(self.device)
                _, ctx_out, _ = self.model.get_ctx_out(ids, attn, ttype, lengths)
                logits        = self.model.actor(ctx_out)
                decoded       = self.model.decode_actions(logits, lengths)
                for i, seq in enumerate(decoded):
                    n = int(lengths[i].item())
                    all_preds.extend(seq[:n])
                    all_trues.extend(labels[i, :n].tolist())

        macro_f1 = f1_score(all_trues, all_preds, average="macro", zero_division=0)
        rare_f1  = f1_score(all_trues, all_preds, labels=list(rare_ids),
                            average="macro", zero_division=0)
        return macro_f1, rare_f1, all_trues, all_preds

    def train(self, train_dataset, dev_dataset, rare_ids, num_epochs=PPO_EPOCHS):
        print(f"\n  Phase D mode : {PHASE_D_MODE}")

        if PHASE_D_MODE in ("curriculum", "combined"):
            self.curriculum = CurriculumScheduler(list(rare_ids))
            print("  ✔ Curriculum scheduler initialised.")

        if PHASE_D_MODE in ("self_play", "combined"):
            print("\n  Collecting self-play correction data ...")
            ctx_dim    = self.model.kg_model.base.ctx_out_dim  # 128
            corr_model = SyntheticCorrectionModel(ctx_dim=ctx_dim)
            corr_data  = collect_correction_data(
                self.model, train_dataset, list(rare_ids), device=self.device)
            self.correction_model = train_correction_model(
                corr_model, corr_data, device=self.device)
            self.correction_model.eval()

        self._optimizer  = self.build_optimizer()
        train_loader     = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                       shuffle=True, collate_fn=collate_rrc)
        loader_iter      = iter(train_loader)
        best_rare_f1     = -1.0
        best_state       = None
        es_counter       = 0
        history          = []

        print(f"\n{'='*60}")
        print("PHASE C+D: PPO RL Training")
        print(f"{'='*60}\n")

        for epoch in range(1, num_epochs + 1):
            epoch_start = time.time()
            buffer, loader_iter = self.collect_rollout(loader_iter, train_loader,
                                                        n_docs=ROLLOUT_DOCS)
            pol_loss, val_loss  = self.ppo_update(buffer)
            macro_f1, rare_f1, all_trues, all_preds = self.evaluate(dev_dataset, rare_ids)

            if self.curriculum is not None:
                self.curriculum.update(all_trues, all_preds)

            epoch_time = time.time() - epoch_start
            print(f"[PPO] Epoch {epoch:03d}/{num_epochs} | "
                  f"pol_loss: {pol_loss:.4f} | val_loss: {val_loss:.4f} | "
                  f"dev_macro_f1: {macro_f1:.4f} | dev_rare_f1: {rare_f1:.4f} | "
                  f"buf: {len(buffer)} | time: {epoch_time:.1f}s")

            if self.curriculum is not None and epoch % 3 == 0:
                self.curriculum.log()

            history.append({
                "epoch": epoch, "phase": "ppo",
                "pol_loss": pol_loss, "val_loss": val_loss,
                "dev_macro_f1": macro_f1, "dev_rare_f1": rare_f1,
                "epoch_time_s": epoch_time,
            })

            if rare_f1 > best_rare_f1 + ES_MIN_DELTA:
                best_rare_f1 = rare_f1
                best_state   = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                es_counter   = 0
                print(f"  ✔ New best dev_rare_f1={best_rare_f1:.4f}")
            else:
                es_counter += 1
                if es_counter >= ES_PATIENCE_PPO:
                    print(f"\n⏹ PPO early stopping at epoch {epoch}.\n"); break

        if best_state is not None:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(OUT_DIR, "ppo_model.bin"))
            print(f"\n✔ Best PPO model saved (dev_rare_f1={best_rare_f1:.4f})")

        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "ppo_history.csv"), index=False)
        return pd.DataFrame(history), best_rare_f1


# ═══════════════════════════════════════════════════════════
# METRICS / VISUALISATION
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec  = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec  = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_rec   = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec   = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)
    acc          = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                               average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)
    per_class_metrics = {id2label[i]: {"f1": float(per_class_f1[i]),
                                        "precision": float(per_class_prec[i]),
                                        "recall": float(per_class_rec[i])}
                         for i in range(NUM_LABELS)}

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_trues  = [id2label[x] for x in all_trues]
    str_preds  = [id2label[x] for x in all_preds]
    cls_report = classification_report(str_trues, str_preds, labels=LABELS,
                                        digits=4, zero_division=0)
    cm         = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {"macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
            "macro_precision": macro_prec, "micro_precision": micro_prec,
            "weighted_precision": weighted_prec,
            "macro_recall": macro_rec, "micro_recall": micro_rec,
            "weighted_recall": weighted_rec,
            "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
            "per_class_metrics": per_class_metrics, "accuracy": acc,
            "cls_report": cls_report, "cm": cm,
            "all_preds": all_preds, "all_trues": all_trues}


def evaluate_ppo_full(ppo_trainer, dataset, rare_ids, split_name="test"):
    macro_f1, rare_f1, all_trues, all_preds = ppo_trainer.evaluate(
        dataset, rare_ids, split_name=split_name)
    acc         = accuracy_score(all_trues, all_preds)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    per_cls_f1  = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                            average=None, zero_division=0)
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]
    report    = classification_report(str_trues, str_preds, labels=LABELS,
                                       digits=4, zero_division=0)
    with open(os.path.join(OUT_DIR, f"ppo_{split_name}_classification_report.txt"), "w") as f:
        f.write(f"Model: InLegalBERT + KG-RAG + PPO (Phase D: {PHASE_D_MODE})\n\n")
        f.write(report)
    return {"accuracy": acc, "macro_f1": macro_f1, "weighted_f1": weighted_f1,
            "rare_f1": rare_f1,
            "per_class_f1": {id2label[i]: float(per_cls_f1[i]) for i in range(NUM_LABELS)},
            "cls_report": report, "all_trues": all_trues, "all_preds": all_preds}


def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels():
            if tick.get_text() in rare_labels: tick.set_color("red")
        for tick in ax.get_yticklabels():
            if tick.get_text() in rare_labels: tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue" for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars    = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12); ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def print_final_table(kg_dev, kg_test, ppo_dev, ppo_test, total_trainable, ppo_time):
    rows = [("Accuracy", "accuracy"), ("Macro-F1", "macro_f1"),
            ("Weighted-F1", "weighted_f1"), ("Rare / Minority F1", "rare_f1")]
    sep = "=" * 80
    print(f"\n{sep}")
    print("FINAL RESULTS  — InLegalBERT + BiLSTM + MHA + CRF + KG-RAG + PPO")
    print(sep)
    print(f"  Trainable parameters : {total_trainable:,}")
    print(f"  Phase C+D training   : {ppo_time/60:.1f} min")
    print(f"  Phase D mode         : {PHASE_D_MODE}")
    print("-" * 80)
    print(f"  {'Metric':<28} {'KG-Dev':>10} {'KG-Test':>10} {'PPO-Dev':>10} {'PPO-Test':>10}")
    print("-" * 80)
    for label, key in rows:
        print(f"  {label:<28} {kg_dev[key]:>10.4f} {kg_test[key]:>10.4f} "
              f"{ppo_dev[key]:>10.4f} {ppo_test[key]:>10.4f}")
    print(sep)
    print("\n  PER-CLASS F1  (Phase B vs Phase C+D — TEST SET)")
    print("  " + "-" * 70)
    print(f"  {'Label':<22} {'KG-RAG':>10} {'PPO':>10} {'Δ':>8}")
    print("  " + "-" * 70)
    for lbl in LABELS:
        kg_f1  = kg_test["per_class_metrics"][lbl]["f1"]
        ppo_f1 = ppo_test["per_class_f1"].get(lbl, 0.0)
        delta  = ppo_f1 - kg_f1
        flag   = " ↑" if delta > 0.005 else (" ↓" if delta < -0.005 else "")
        print(f"  {lbl:<22} {kg_f1:>10.4f} {ppo_f1:>10.4f} {delta:>+8.4f}{flag}")
    print("  " + "-" * 70)


# ═══════════════════════════════════════════════════════════
# MAIN — Resumes from Phase C+D
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device  : {DEVICE}")
    print(f"Resuming from Phase C+D (Phase D mode: {PHASE_D_MODE})\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading data ...")
    train_docs = extract_docs(load_jsonl(TRAIN_PATH))
    dev_docs   = extract_docs(load_jsonl(DEV_PATH))
    test_docs  = extract_docs(load_jsonl(TEST_PATH))
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    print(f"  Rare classes ({len(rare_labels)}): {rare_labels}")

    print("Loading tokenizer ...")
    tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ── Load saved base model ──────────────────────────────
    # AutoModel.from_pretrained needs pytorch_model.bin / model.safetensors,
    # but we saved under a custom name (base_model.bin).
    # Solution: initialise architecture from the original HF hub,
    # then overlay ALL weights from our saved checkpoint.
    print("\nLoading saved base model (Phase A) ...")
    base_model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name=INLEGALBERT_MODEL_NAME,   # <-- hub name, not local dir
        sent_lstm_hidden=SENT_LSTM_HIDDEN, sent_lstm_layers=SENT_LSTM_LAYERS,
        ctx_lstm_hidden=CTX_LSTM_HIDDEN,   ctx_lstm_layers=CTX_LSTM_LAYERS,
        mha_heads=MHA_HEADS, mha_dropout=MHA_DROPOUT,
        num_labels=NUM_LABELS, dropout=DROPOUT, freeze_layers=BERT_FREEZE_LAYERS)

    base_state = torch.load(os.path.join(BEST_MODEL_DIR, "base_model.bin"),
                             map_location="cpu")
    missing, unexpected = base_model.load_state_dict(base_state, strict=False)
    if missing:
        print(f"  ⚠ Missing keys  ({len(missing)}): {missing[:5]} ...")
    if unexpected:
        print(f"  ⚠ Unexpected keys ({len(unexpected)}): {unexpected[:5]} ...")
    base_model.to(DEVICE)
    print("  ✔ Base model loaded from base_model.bin.")

    # ── Load KG ───────────────────────────────────────────
    print("\nLoading Knowledge Graph ...")
    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    kg      = KnowledgeGraph.load(kg_path, emb_dim=base_model.sent_out_dim)
    retriever = KGRetriever(kg, top_k=KG_TOP_K, top_nodes=KG_TOP_NODES, hop=KG_HOP)

    # ── Build KGAugmentedModel and load Phase B weights ───
    print("\nLoading KG-Augmented model (Phase B) ...")
    kg_model   = KGAugmentedModel(base_model=base_model, kg=kg,
                                   rare_ids=rare_ids, retriever=retriever)
    kg_state   = torch.load(os.path.join(BEST_MODEL_DIR, "kg_model.bin"),
                             map_location="cpu")
    kg_model.load_state_dict(kg_state, strict=False)
    kg_model.to(DEVICE)
    print("  ✔ KG model loaded.")

    # ── Re-evaluate Phase B for comparison table ──────────
    print("\nRe-evaluating Phase B (KG-RAG) on dev + test ...")
    from torch.utils.data import DataLoader as DL

    def quick_eval_kg(kg_m, dataset, rare_ids_list):
        kg_m.eval()
        loader = DL(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_p, all_t = [], []
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(DEVICE); attn = attn.to(DEVICE)
                ttype = ttype.to(DEVICE); lengths = lengths.to(DEVICE)
                decoded, _ = kg_m(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq in enumerate(decoded):
                    n = int(lengths[i].item())
                    all_p.extend(seq); all_t.extend(labels[i, :n].tolist())
        return compute_all_metrics(all_t, all_p, rare_ids_list)

    kg_dev_metrics  = quick_eval_kg(kg_model, dev_dataset,  rare_ids)
    kg_test_metrics = quick_eval_kg(kg_model, test_dataset, rare_ids)
    print(f"  [KG] Dev  Macro-F1: {kg_dev_metrics['macro_f1']:.4f}  "
          f"Rare-F1: {kg_dev_metrics['rare_f1']:.4f}")
    print(f"  [KG] Test Macro-F1: {kg_test_metrics['macro_f1']:.4f}  "
          f"Rare-F1: {kg_test_metrics['rare_f1']:.4f}")

    total_trainable = sum(p.numel() for p in kg_model.parameters() if p.requires_grad)

    # ══════════════════════════════════════════════════════
    # PHASE C+D — PPO RL + Synthetic Correction
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE C+D: PPO RL + Synthetic Correction")
    print("=" * 60)

    ppo_model   = PPOModel(kg_model=kg_model, rare_ids=rare_ids)
    ppo_trainer = PPOTrainer(ppo_model=ppo_model, rare_ids=rare_ids,
                              kg_retriever=retriever, device=DEVICE)

    ppo_start = time.time()
    ppo_hist_df, best_rare_f1 = ppo_trainer.train(
        train_dataset, dev_dataset, rare_ids, num_epochs=PPO_EPOCHS)
    ppo_time = time.time() - ppo_start

    # ── Evaluation ────────────────────────────────────────
    print("\nEvaluating Phase C+D on Dev set ...")
    ppo_dev_metrics  = evaluate_ppo_full(ppo_trainer, dev_dataset,  rare_ids, "ppo_dev")
    print(f"  [PPO] Dev  Macro-F1: {ppo_dev_metrics['macro_f1']:.4f}  "
          f"Rare-F1: {ppo_dev_metrics['rare_f1']:.4f}")

    print("\nEvaluating Phase C+D on Test set ...")
    ppo_test_metrics = evaluate_ppo_full(ppo_trainer, test_dataset, rare_ids, "ppo_test")
    print(f"  [PPO] Test Macro-F1: {ppo_test_metrics['macro_f1']:.4f}  "
          f"Rare-F1: {ppo_test_metrics['rare_f1']:.4f}")

    # ── Confusion matrices & charts ───────────────────────
    ppo_cm = confusion_matrix(
        [id2label[x] for x in ppo_test_metrics["all_trues"]],
        [id2label[x] for x in ppo_test_metrics["all_preds"]],
        labels=LABELS)
    save_confusion_matrix(ppo_cm, "ppo_test", rare_labels)

    ppo_pcm = {lbl: {"f1": ppo_test_metrics["per_class_f1"].get(lbl, 0.0),
                     "precision": 0.0, "recall": 0.0} for lbl in LABELS}
    save_per_class_f1_chart(ppo_pcm, "ppo_test", rare_labels)

    # ── Prediction CSV ────────────────────────────────────
    pd.DataFrame({
        "true": [id2label[x] for x in ppo_test_metrics["all_trues"]],
        "pred": [id2label[x] for x in ppo_test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "ppo_test_predictions.csv"), index=False)

    # ── Training curves ───────────────────────────────────
    if os.path.exists(os.path.join(OUT_DIR, "base_history.csv")):
        base_hist_df = pd.read_csv(os.path.join(OUT_DIR, "base_history.csv"))
    else:
        base_hist_df = pd.DataFrame()
    if os.path.exists(os.path.join(OUT_DIR, "kg_history.csv")):
        kg_hist_df = pd.read_csv(os.path.join(OUT_DIR, "kg_history.csv"))
    else:
        kg_hist_df = pd.DataFrame()

    ppo_hist_rename = ppo_hist_df.rename(columns={
        "dev_macro_f1": "val_macro_f1", "dev_rare_f1": "val_rare_f1",
        "pol_loss": "train_loss"})

    if not base_hist_df.empty and not kg_hist_df.empty:
        all_df = pd.concat([base_hist_df, kg_hist_df, ppo_hist_rename], ignore_index=True)
        all_df["global_epoch"] = range(1, len(all_df)+1)
        b1 = len(base_hist_df); b2 = b1 + len(kg_hist_df)
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        for ax, col, title in zip(axes,
            ["train_loss", "val_macro_f1"],
            ["Combined Training Loss (A→B→C+D)", "Val F1 (Base→KG-RAG→PPO+RL)"]):
            ax.plot(all_df["global_epoch"], all_df[col].fillna(method="ffill"),
                    marker="o", markersize=2, label=col)
            ax.axvline(b1, color="orange", linestyle="--", label="Phase B start")
            ax.axvline(b2, color="purple", linestyle="--", label="Phase C start")
            if "f1" in col:
                ax.plot(all_df["global_epoch"],
                        all_df.get("val_rare_f1", pd.Series()).fillna(method="ffill"),
                        marker="s", markersize=2, label="Val Rare-F1")
            ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        path = os.path.join(OUT_DIR, "all_phases_training_curves.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")

    # ── Final JSON summary ─────────────────────────────────
    scalar_keys = ["macro_f1", "micro_f1", "weighted_f1", "rare_f1",
                   "macro_precision", "micro_precision", "weighted_precision",
                   "macro_recall", "micro_recall", "weighted_recall",
                   "rare_precision", "rare_recall", "accuracy"]
    summary = {
        "model": "InLegalBERT + BiLSTM + MHA + CRF + KG-RAG + PPO",
        "phase_d_mode": PHASE_D_MODE,
        "rare_classes": rare_labels,
        "timing": {"phase_cd_s": ppo_time},
        "kg_rag_dev":   {k: kg_dev_metrics[k]  for k in scalar_keys if k in kg_dev_metrics},
        "kg_rag_test":  {k: kg_test_metrics[k] for k in scalar_keys if k in kg_test_metrics},
        "ppo_dev":      {k: ppo_dev_metrics[k]  for k in scalar_keys if k in ppo_dev_metrics},
        "ppo_test":     {k: ppo_test_metrics[k] for k in scalar_keys if k in ppo_test_metrics},
        "per_class_kg_test":  kg_test_metrics["per_class_metrics"],
        "per_class_ppo_test": ppo_test_metrics["per_class_f1"],
    }
    with open(os.path.join(OUT_DIR, "full_metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print_final_table(kg_dev_metrics, kg_test_metrics,
                      ppo_dev_metrics, ppo_test_metrics,
                      total_trainable, ppo_time)
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device  : cuda:0
Resuming from Phase C+D (Phase D mode: combined)

Loading data ...
  Train: 245 | Dev: 30 | Test: 50
  Rare classes (10): ['RLC', 'ISSUE', 'ARG_PETITIONER', 'ARG_RESPONDENT', 'STA', 'PRE_RELIED', 'PRE_NOT_RELIED', 'RATIO', 'RPC', 'NONE']
Loading tokenizer ...

Loading saved base model (Phase A) ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✔ Base model loaded from base_model.bin.

Loading Knowledge Graph ...
  KG loaded from rrc_kg_rag_ppo_logs/knowledge_graph.json

Loading KG-Augmented model (Phase B) ...
  ✔ KG model loaded.

Re-evaluating Phase B (KG-RAG) on dev + test ...
  [KG] Dev  Macro-F1: 0.6160  Rare-F1: 0.5305
  [KG] Test Macro-F1: 0.6922  Rare-F1: 0.6288

PHASE C+D: PPO RL + Synthetic Correction
  ✔ ActorHead warm-started from Phase B fusion_classifier.

  Phase D mode : combined
  ✔ Curriculum scheduler initialised.

  Correction data: 1330 errors, 5835 confident-correct
    [Correction] epoch 5/15 loss=0.9955
    [Correction] epoch 10/15 loss=0.9589
    [Correction] epoch 15/15 loss=0.9293
  ✔ Correction model saved.

PHASE C+D: PPO RL Training

[PPO] Epoch 001/20 | pol_loss: 0.0333 | val_loss: 1109.8080 | dev_macro_f1: 0.6168 | dev_rare_f1: 0.5314 | buf: 1582 | time: 706.3s
  ✔ New best dev_rare_f1=0.5314
[PPO] Epoch 002/20 | pol_loss: 0.0067 | val_loss: 1249.2269 | dev_macro_f1: 0.6197 | dev_rare_f1: 0.

/tmp/ipykernel_132422/3465901843.py:1384: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  ax.plot(all_df["global_epoch"], all_df[col].fillna(method="ffill"),
/tmp/ipykernel_132422/3465901843.py:1390: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  all_df.get("val_rare_f1", pd.Series()).fillna(method="ffill"),


Saved rrc_kg_rag_ppo_logs/all_phases_training_curves.png

FINAL RESULTS  — InLegalBERT + BiLSTM + MHA + CRF + KG-RAG + PPO
  Trainable parameters : 30,934,048
  Phase C+D training   : 67.3 min
  Phase D mode         : combined
--------------------------------------------------------------------------------
  Metric                           KG-Dev    KG-Test    PPO-Dev   PPO-Test
--------------------------------------------------------------------------------
  Accuracy                         0.8225     0.8437     0.8218     0.8415
  Macro-F1                         0.6160     0.6922     0.6197     0.6945
  Weighted-F1                      0.8109     0.8448     0.8113     0.8444
  Rare / Minority F1               0.5305     0.6288     0.5356     0.6320

  PER-CLASS F1  (Phase B vs Phase C+D — TEST SET)
  ----------------------------------------------------------------------
  Label                      KG-RAG        PPO        Δ
  ------------------------------------------------------